# AKANE E15：鎖の最後の段の判定と、親の語の選び方

E14 で、正しいのに「いいえ」と言っていた問題の多くは、**正しい途中の語（例 子猫→猫、妻→人間）まで着いているのに、最後の「分類に着く」判定で落ちていた**ことが分かりました。今の判定は、その語の辞書の説明文1つで NLI に聞いています（人間の説明文に「哺乳類」と書いてないと落ちる）。

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：plans/E15_last_step.md。

- **最後の段**：たどった道の説明文を全部見せて、Gemma に「この説明文から、x は C の一種ですか」と聞きます。対照として、道の最後の説明文だけ・関係のない別の問題の道・説明文なし、でも聞きます。
- **親の語**：説明文の中のどの語が「〇〇の一種」の〇〇に当たるかを、Gemma に選ばせます（今までは NLI の値で選んでいた）。
- 問題は2組：E8 の 1,000問（直し方を見る）と、E10-3 の「一種か」600問（確かめ。501語×46分類から常識で採点した600問）。
- 正解はこのノートに入っていません。手で書いた語の一覧もありません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（今までと同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E15_results/E15_results.json.gz` ができます。
5. 時間の見込み：40〜70分。途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所は飛ばします。

In [ ]:
import os, json, time, subprocess, sys, gzip, collections, re, random
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E15_results"
os.makedirs(OUT, exist_ok=True)
E112 = f"{BASE}/E11-2_results/E11-2_results.json"
WIKT = f"{BASE}/E10-1_results/E101_wiktionary.json.gz"   # E10-1 で読んだ名詞の辞書
for p in (E112, WIKT):
    assert os.path.exists(p), "見つかりません: " + p
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E6〜E14 と同じ
GEMMA_ID = "google/gemma-2-2b-it"
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)
def save_gz(obj, name):
    p = f"{OUT}/{name}.json.gz"
    with gzip.open(p, "wt", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False)
    print("保存:", name, round(os.path.getsize(p) / 1e6, 2), "MB"); return p
def load_gz(name):
    p = f"{OUT}/{name}.json.gz"
    if not os.path.exists(p): return None
    with gzip.open(p, "rt", encoding="utf-8") as f: return json.load(f)

In [ ]:
# 中身1（analysis/nolist/e8n_core.py と同じ。手書きの語の一覧はありません）
"""E8の中身（Colabと手元の両方で使う）。
E7との違い：
- ダンプを読む時に、意味ごとの印（{{lb|en|slang}} など）を残す。途中の語の意味は8個まで持つ。
- 鎖の点を「語」ではなく「語＋意味の番号」にし、途中の語の意味を選ぶ方法を4通り：
  A 選ばない（E7と同じ、途中の語は最初の4つの意味を全部）
  B 印（古い・まれ・俗語など）のない最初の意味（Kovács 2022型）
  C 1つ前の説明文に一番近い意味（文の近さ、McIntosh 2009・OntoLearn型。分類Cは入れない）
  D Gemmaに「この説明文の中の語 g はどの意味か」を選ばせる（Meconi 2025・ConSeC型。分類Cは入れない）
- 各点から、46分類すべてへの「着き」をNLIで測る（打ち消し＝両立しない分類に着くか、に使う）。
"""
import re

MAX_DEFS_ENT, MAX_DEFS_MID, A_MID, DEPTH = 8, 8, 4, 3
# 一覧なし版（2026-11-11、本人「ハードコードは絶対ダメ」）：手で書いた語の一覧（印・頭の名詞の決まり）を使わない。
# 頭の名詞は文法（spaCyの名詞のかたまりと係り受け）だけで候補を出し、どれを使うかはNLIの段の値で決める（Colab側 set_genus）。


def genus_candidates(defn, nlp, max_c=4):
    """最初の名詞のかたまりの頭、その頭に付いた of の目的語（あれば）。それぞれ見出し形と、そのままの形（違う時）。名詞・固有名詞だけ。"""
    t = re.sub(r"\([^)]*\)", " ", defn).strip()
    doc = nlp(t[:200])
    chunks = [c for c in doc.noun_chunks if c.root.pos_ in ("NOUN", "PROPN")]
    if not chunks:
        return []
    root = chunks[0].root
    toks = [root] + [o for ch in root.children if ch.dep_ == "prep" and ch.text.lower() == "of"
                     for o in ch.children if o.dep_ == "pobj" and o.pos_ in ("NOUN", "PROPN")]
    out = []
    for tk in toks:
        for w in (tk.lemma_.lower(), tk.text.lower()):
            if w not in out and re.fullmatch(r"[a-z][a-z\-]*", w):
                out.append(w)
    return out[:max_c]


# --- ダンプの読み取り（E7と同じ書き換え。ただし lb の印は取っておく） ---
DROP_T = {"lb", "lbl", "label", "q", "qual", "qualifier", "i", "gloss", "gl", "sense", "s", "defdate", "rfdef", "rfclarify",
          "senseid", "tlb", "def-date", "c", "C", "topics", "cat", "rfex", "rfquote", "nb...", "ux", "uxi", "quote-book"}
ARG2_T = {"l", "m", "link", "mention", "ll", "l-self", "m-self", "cog"}
ARG1_T = {"w", "taxlink", "taxfmt", "vern", "ws", "pedia", "non-gloss", "n-g", "ngd", "non-gloss definition", "glossary",
          "smallcaps", "taxon", "spelink", "nowrap", "small", "sub", "sup"}
SKIP_LINE = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of", "misspelling of",
             "abbreviation of", "initialism of", "acronym of", "obsolete spelling of", "archaic spelling of", "eye dialect of",
             "nonstandard spelling of", "synonym of", "clipping of", "short for", "ellipsis of", "contraction of")
SKIP_T = re.compile(r"^\{\{\s*(plural of|alternative form of|alt form|alt sp|alternative spelling of|obsolete form of|"
                    r"misspelling of|abbreviation of|abbr of|initialism of|acronym of|obsolete spelling of|archaic spelling of|"
                    r"eye dialect of|nonstandard spelling of|synonym of|syn of|clipping of|short for|ellipsis of|"
                    r"contraction of|form of|inflection of|en-past of)\s*\|", re.I)
LB = re.compile(r"\{\{\s*(?:lb|lbl|label|tlb)\s*\|\s*en\s*\|([^{}]*)\}\}")


def _tmpl(m):
    parts = [p.strip() for p in m.group(1).split("|")]
    name, args = parts[0].strip(), [p for p in parts[1:] if "=" not in p]
    if name in DROP_T:
        return ""
    if name in ARG2_T:
        return args[1] if len(args) > 1 else ""
    if name in ARG1_T:
        return args[0] if args else ""
    return ""


def strip_wikitext(t):
    for _ in range(6):
        t2 = re.sub(r"\{\{([^{}]*)\}\}", _tmpl, t)
        if t2 == t:
            break
        t = t2
    t = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", t)
    t = re.sub(r"'{2,}|<[^>]+>", "", t)
    return re.sub(r"\s+", " ", t).strip(" ;,.") + "."


def labels_of(raw):
    out = []
    for m in LB.finditer(raw):
        for p in m.group(1).split("|"):
            p = p.strip().lower()
            if p and "=" not in p and p not in ("_", "and", "or"):
                out.append(p)
    return out


def noun_senses_from_wikitext(wt, limit=MAX_DEFS_MID):
    """[[説明文, [印]], ...]（E7と同じ説明文の並び。印は {{lb|en|...}} から）"""
    m = re.search(r"(?ms)^==English==\s*$(.*?)(?=^==[^=]|\Z)", wt or "")
    if not m:
        return []
    out = []
    for sec in re.finditer(r"(?ms)^===+\s*Noun\s*===+\s*$(.*?)(?=^===|\Z)", m.group(1)):
        for line in sec.group(1).splitlines():
            if not line.startswith("# "):
                continue
            raw = line[2:].strip()
            if SKIP_T.match(raw):
                continue
            d = strip_wikitext(raw)
            if len(d) > 3 and not d.lower().startswith(SKIP_LINE):
                out.append([d[:220], labels_of(raw)])
    return out[:limit]


def parse_dump(fobj, log=print, every=500000):
    import xml.etree.ElementTree as ET
    out, n, title, ns = {}, 0, None, None
    for ev, el in ET.iterparse(fobj, events=("end",)):
        tag = el.tag.rsplit("}", 1)[-1]
        if tag == "title":
            title = el.text
        elif tag == "ns":
            ns = el.text
        elif tag == "text":
            if ns == "0" and title and re.fullmatch(r"[a-z][a-z\-]*", title) and el.text and "==English==" in el.text:
                ds = noun_senses_from_wikitext(el.text)
                if ds:
                    out[title] = ds
        elif tag == "page":
            n += 1
            el.clear()
            if n % every == 0:
                log(f"ページ {n:,}  名詞のある語 {len(out):,}")
    return out


# --- 鎖をたどる ---
def build_lexicon(items, WKD, nlp, log=print):
    """L[w] = [{"def", "labels", "genus"}]。物の語は items の wk_defs（印なし）か WKD、途中の語は WKD から8個まで。
    A用の段（途中の語は最初の4つ全部）と、B/C/D用の段（途中の語は8個から1つ選ぶ）の両方で出てくる語をすべて集める。"""
    L = {}
    def senses(w, ent_defs=None):
        if w in L:
            return L[w]
        if ent_defs is not None:
            ss = [[d, []] for d in ent_defs][:MAX_DEFS_ENT]
        else:
            ss = WKD.get(w, [])[:MAX_DEFS_MID]
        L[w] = [{"def": d, "labels": lb, "cands": genus_candidates(d, nlp), "genus": None} for d, lb in ss]
        return L[w]
    front = set()
    for it in items:
        ed = it.get("wk_defs")
        senses(it["ent"], ed if ed else None)
        front.add(it["ent"])
    for depth in range(DEPTH):
        nxt = set()
        for w in front:
            for x in L[w]:
                for g in x["cands"]:
                    if g not in L:
                        senses(g)
                        nxt.add(g)
        log(f"段{depth + 1}: 新しい語 {len(nxt)}  語の合計 {len(L)}")
        front = nxt
    return L


def set_genus(L, SC):
    """SC[(w,k,g)] = NLIの「含意」（"w" means: d ⇒ W is a type of g.）。候補のうち辞書にあり、一番値の大きい物を頭の名詞にする（一覧なし）。"""
    for w, xs in L.items():
        for k, x in enumerate(xs):
            cs = [g for g in x["cands"] if g != w and L.get(g)]
            x["genus"] = max(cs, key=lambda g: SC.get((w, k, g), 0.0)) if cs else None
    return L


def edges(L, ents):
    """(w,k) で genus g を持つ段のうち、たどり得るもの（物の語から3段まで、途中の語は全意味）"""
    seen, front, out = set(ents), set(ents), set()
    for _ in range(DEPTH):
        nxt = set()
        for w in front:
            for k, x in enumerate(L.get(w, [])):
                g = x["genus"]
                if g and g in L and g != w:
                    out.add((w, k))
                    if g not in seen:
                        seen.add(g); nxt.add(g)
        front = nxt
    return sorted(out)


def choice_prompt(w, d, g, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Dictionary definition of \"{w}\": {d}\n\nIn this definition, which meaning of the word \"{g}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def premise(w, d):
    return f"\"{w}\" means: {d}"


def hyp(w, c):
    return f"{w[:1].upper() + w[1:]} is a type of {c}."




def chain_nodes(L, x, method, choice=None):
    """物の語 x から3段までの点（語, 意味の番号）と段（(w,k)→(g,j)）。
    A：途中の語は最初の4つの意味を全部（E7と同じ）。B/C/D：途中の語は choice[(w,k)] で1つに決める。物の語は全部の意味。"""
    nodes, links = set(), set()
    front = [(x, k) for k in range(len(L.get(x, [])))]
    nodes.update(front)
    for _ in range(DEPTH):
        nxt = []
        for (w, k) in front:
            g = L[w][k]["genus"]
            if not g or g not in L or g == w or not L[g]:
                continue
            if method == "A":
                js = range(min(A_MID, len(L[g])))
            else:
                js = [choice.get((w, k), 0) if len(L[g]) > 1 else 0]
            for j in js:
                links.add(((w, k), (g, j)))
                if (g, j) not in nodes:
                    nodes.add((g, j)); nxt.append((g, j))
        front = nxt
    return nodes, links

In [ ]:
# 中身2（analysis/e14/e14_core.py と同じ。new_word だけ使います）
"""E14 の中身：辞書の鎖（E11-2 の6段の鎖＝E8n 方法D を伸ばした物）の各点に、
別の資料の「A は B の一種」を継ぎ足す（計画 plans/E14_fill_gaps.md）。手書きの語の一覧はない。
e8n_core（genus_candidates, choice_prompt, premise, hyp, MAX_DEFS_MID）の後に置いて使う。
NLI・Gemma・文の資料は呼び出しの関数として外から渡す（Colab ではモデル、手元の確かめでは作り物）。

点の種類：
  ("s", w, k)  辞書の意味（E8n と同じ。L[w][k]）
  ("g", w, i)  継ぎ足しの文（GR[w][i]。src は "g"＝一般的な文、"s"＝Gemma が書いた文）
つながり：
  ("s",w,k) → ("s",g,j)   辞書の頭の語（強さ ST[(w,k)]、1段）
  ("s",w,k) → ("g",w,i)   同じ語の文。文の中の w の意味が k の時だけ（Gemma が選ぶ）。物の語の文は根から直接。強さ1、段に数えない
  ("g",w,i) → ("s",p,j)   文の中の親の語 p（強さ＝NLI「文 ⇒ W is a type of p.」、1段）。p の意味 j は Gemma が選ぶ
"""
import re

MAX_DEPTH = 6        # E11-2 と同じ上限
K_GKB = 3            # 一般的な文は1語につき3文まで（親への NLI の値が高い順）
N_GKB_READ = 20      # 1語につき読む一般的な文（資料の score の高い順）
MAX_CANDS = 6        # 1文の親の候補の数
SELF_TOKENS = 40     # 書かせる長さ（30語ほど）
WORD = re.compile(r"[a-z][a-z\-]*")


def self_prompt(w):
    return f"Write one short sentence saying what kind of thing a \"{w}\" is."


def sense_in_sentence_prompt(sent, w, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Sentence: {sent}\n\nIn this sentence, which meaning of the word \"{w}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def clean_self(text):
    """書かせた文：最初の空でない行。行の頭の記号を外す（書式の読み方）。"""
    for line in text.splitlines():
        s = re.sub(r"^[\s\-\*\d\.\)\"]+", "", line).strip()
        if len(s.split()) >= 3:
            return s
    return None


def subject_parents(doc, w, alias=None, pron=False):
    """文の主語（名詞のかたまりで、係りが主語）の頭が w の時、文の中のほかの名詞のかたまりの頭（見出し形）を親の候補にする。
    w が主語でなければ None。どの候補を親にするかは NLI で決める（ここでは決めない）。
    E14-3：pron=True なら代名詞の主語（品詞が PRON）も認める（見出しが w と同じ語の記事の段落の中の文だけで使う）。alias は使っていない（転送先の別の語、例 inventor→Invention を主語にしないため）。"""
    chunks = list(doc.noun_chunks)
    names = {w} | ({alias.lower()} if alias else set())
    subj = [c for c in chunks if c.root.dep_ in ("nsubj", "nsubjpass")
            and (names & {c.root.lemma_.lower(), c.root.text.lower()} or (pron and c.root.pos_ == "PRON"))]
    if not subj:
        return None
    s = subj[0]
    out = []
    for c in chunks:
        if c.start == s.start:
            continue
        if c.root.pos_ != "NOUN":
            continue
        lw = c.root.lemma_.lower()
        if lw != w and WORD.fullmatch(lw) and lw not in out:
            out.append(lw)
    return out[:MAX_CANDS]


def new_word(L, w, WKD, nlp):
    """辞書にまだ無い語を WKD（E10-1 の名詞の辞書）から読む。頭の名詞はまだ決めない。"""
    if w in L:
        return bool(L[w])
    L[w] = [{"def": d, "labels": lb, "cands": genus_candidates(d, nlp), "genus": None}
            for d, lb in WKD.get(w, [])[:MAX_DEFS_MID]]
    return bool(L[w])


def follow(L, choice, GR, x, maxd, srcs=("g", "s")):
    """物の語 x から maxd 段まで。点 → 一番浅い段の数、つながりの組。srcs で使う資料を選ぶ（() なら E11-2 の鎖と同じ）。"""
    depth, links = {}, set()
    front = []
    for k in range(len(L.get(x, []))):
        depth[("s", x, k)] = 0; front.append(("s", x, k))
    for i, g in enumerate(GR.get(x, [])):
        if g["src"] in srcs and g.get("parent"):
            depth[("g", x, i)] = 0; front.append(("g", x, i))
    d = 0
    while front and d < maxd:
        nxt = []
        for n in front:
            outs = []
            if n[0] == "s":
                _, w, k = n
                gw = L[w][k].get("genus")
                if gw and gw in L and gw != w and L[gw]:
                    j = choice.get((w, k), 0) if len(L[gw]) > 1 else 0
                    outs.append((("s", gw, j), 1))
                if w != x:   # 途中の語の文（意味が合う時だけ、段に数えない）
                    for i, g in enumerate(GR.get(w, [])):
                        if g["src"] in srcs and g.get("parent") and g.get("wsense", 0) == k:
                            outs.append((("g", w, i), 0))
            else:
                _, w, i = n
                g = GR[w][i]
                outs.append((("s", g["parent"], g.get("psense", 0)), 1))
            for m, step in outs:
                links.add((n, m))
                dm = depth[n] + step
                if m not in depth or dm < depth[m]:
                    depth[m] = dm
                    if step == 0:
                        front.append(m) if dm == d else nxt.append(m)
                    else:
                        nxt.append(m)
        front = nxt
        d += 1
    return depth, links


def grow(items, L, choice, ST, AR, GR, ARG, WKD, nlp, nli, nli_sent, gemma_choice, gemma_sense, gkb_sents, self_write,
         maxd=MAX_DEPTH, log=print, wiki_sents=None):
    """1段ずつ広げる。（E14-2：wiki_sents([語])→{語: 文 or None} を渡すと、Wikipedia の最初の1文を src "w" として足す）各段で、先の点の語について：
    a. 文を集める（一般的な文 N_GKB_READ 文・書かせた文1文）→ 主語が w の文だけ、親の候補を出す
    b. 親の候補を NLI（文 ⇒ W is a type of 候補）で測り、一番の候補を親に。一般的な文は値の高い順に K_GKB 文
    c. 親の語を辞書から読み、親の意味・文の中の w の意味を Gemma が選ぶ
    d. 辞書の意味の点は E11-2 と同じく 頭の名詞・段の値・意味選び（新しい語だけ）
    e. 新しく着いた点の、問題の分類への着き（辞書の意味は E8n と同じ NLI、文は 文 ⇒ W is a type of C.）
    関数：nli([(w,k,c)])→[(含意,矛盾)]（"w" means: 説明文 ⇒ W is a type of c.）、nli_sent([(文, w, c)])→[(含意,矛盾)]、
    gemma_choice([(w,k)])→[番号]（E8n の方法D）、gemma_sense([(文, 語, [説明文])])→[番号]、
    gkb_sents([語])→{語: [文,...]}（score の高い順）、self_write([語])→{語: 文 or None}。"""
    stat = {"words_grafted": 0, "gkb_sent_read": 0, "gkb_kept": 0, "self_kept": 0, "new_words": 0, "arrive": 0}
    done = set(GR)
    n0 = len(L)
    for lev in range(maxd):
        fr = set()
        for it in items:
            dep, _ = follow(L, choice, GR, it["ent"], lev)
            fr |= {n for n, d in dep.items() if d == lev}
        words = sorted({n[1] for n in fr if n[0] == "s"} - done)
        # a. 文を集める
        G = gkb_sents(words) if words else {}
        S = self_write(words) if words else {}
        Wk = wiki_sents(words) if (words and wiki_sents) else {}
        cand = []   # (w, src, 文, [候補])
        for w in words:
            for s in G.get(w, [])[:N_GKB_READ]:
                ps = subject_parents(nlp(s), w)
                if ps:
                    cand.append((w, "g", s, ps))
            stat["gkb_sent_read"] += len(G.get(w, [])[:N_GKB_READ])
            for src, s in (("s", S.get(w)), ("w", Wk.get(w))):
                # E14-3：wiki_sents が {語: (記事の見出しが w と同じ語か, [文,...])} を返す時は、段落の文を全部見る。
                # 主語は w。代名詞（It・They など）の主語は、記事の見出しが w と同じ語の時だけ認める（転送先の別の語の記事では、代名詞は別の物を指すため）
                if isinstance(s, tuple):
                    same, sl = s
                    for x in sl:
                        ps = subject_parents(nlp(x), w, pron=bool(same))
                        if ps:
                            cand.append((w, src, x, ps))
                elif s:
                    ps = subject_parents(nlp(s), w)
                    if ps:
                        cand.append((w, src, s, ps))
        # 親の候補は辞書にある語だけ
        cand = [(w, src, s, [p for p in ps if new_word(L, p, WKD, nlp)]) for w, src, s, ps in cand]
        cand = [c for c in cand if c[3]]
        # b. 親を NLI で決める
        trip = sorted({(s, w, p) for w, src, s, ps in cand for p in ps})
        sc = dict(zip(trip, nli_sent(trip))) if trip else {}
        per = {}
        for w, src, s, ps in cand:
            best = max(ps, key=lambda p: sc[(s, w, p)][0])
            per.setdefault((w, src), []).append({"src": src, "sent": s, "parent": best, "step": sc[(s, w, best)],
                                                 "cands": ps})
        for w in words:
            gs = sorted(per.get((w, "g"), []), key=lambda g: -g["step"][0])[:K_GKB]
            ss = per.get((w, "s"), [])[:1]
            ws = sorted(per.get((w, "w"), []), key=lambda g: -g["step"][0])[:1]   # E14-2 は1文だけなので並べても同じ
            GR[w] = gs + ss + ws
            done.add(w)
            stat["words_grafted"] += 1; stat["gkb_kept"] += len(gs); stat["self_kept"] += len(ss)
        # c. 親の意味と、文の中の w の意味（意味が2つ以上の時だけ Gemma）
        jobs, keys = [], []
        for w in words:
            for i, g in enumerate(GR[w]):
                p = g["parent"]
                if len(L[p]) > 1:
                    jobs.append((g["sent"], p, [x["def"] for x in L[p]])); keys.append((w, i, "psense"))
                else:
                    g["psense"] = 0
                if len(L.get(w, [])) > 1:
                    jobs.append((g["sent"], w, [x["def"] for x in L[w]])); keys.append((w, i, "wsense"))
                else:
                    g["wsense"] = 0
        if jobs:
            for (w, i, f), v in zip(keys, gemma_sense(jobs)):
                GR[w][i][f] = int(v)
        # d. 辞書の意味の点：新しい語の頭の名詞・段の値・意味選び（E11-2 の extend と同じ手順）
        dep_all = set()
        for it in items:
            dep, _ = follow(L, choice, GR, it["ent"], lev + 1)
            dep_all |= set(dep)
        sn = sorted(n for n in dep_all if n[0] == "s" and L[n[1]][n[2]].get("genus") is None and L[n[1]][n[2]]["cands"]
                    and (n[1], n[2]) not in ST)
        for _, w, k in sn:
            for c in L[w][k]["cands"]:
                if c != w and c not in L:
                    stat["new_words"] += int(new_word(L, c, WKD, nlp))
        pairs = sorted({(w, k, c) for _, w, k in sn for c in L[w][k]["cands"] if c != w and L.get(c)})
        gsc = dict(zip(pairs, nli(pairs))) if pairs else {}
        for _, w, k in sn:
            cs = [c for c in L[w][k]["cands"] if (w, k, c) in gsc]
            if cs:
                L[w][k]["genus"] = max(cs, key=lambda c: gsc[(w, k, c)][0])
                ST[(w, k)] = gsc[(w, k, L[w][k]["genus"])]
        cj = sorted((n[1], n[2]) for n in dep_all if n[0] == "s" and L[n[1]][n[2]].get("genus")
                    and len(L.get(L[n[1]][n[2]]["genus"], [])) > 1 and (n[1], n[2]) not in choice)
        if cj:
            for n, v in zip(cj, gemma_choice(cj)):
                choice[n] = int(v)
        # e. 着き
        dep_all = {}
        arr_s, arr_g = set(), set()
        for it in items:
            dep, _ = follow(L, choice, GR, it["ent"], lev + 1)
            C = it["catw"]
            for n in dep:
                if n[0] == "s" and (n[1], n[2], C) not in AR:
                    arr_s.add((n[1], n[2], C))
                elif n[0] == "g" and (n[1], n[2], C) not in ARG:
                    arr_g.add((n[1], n[2], C))
        arr_s, arr_g = sorted(arr_s), sorted(arr_g)
        if arr_s:
            for n, v in zip(arr_s, nli(arr_s)):
                AR[n] = v
        if arr_g:
            for n, v in zip(arr_g, nli_sent([(GR[w][i]["sent"], w, C) for w, i, C in arr_g])):
                ARG[n] = v
        stat["arrive"] += len(arr_s) + len(arr_g)
        log(f"段{lev + 1}: 先の点 {len(fr)}  文を探した語 {len(words)}  文の候補 {len(cand)}  新しい語 {stat['new_words']}  "
            f"着き {len(arr_s)}+{len(arr_g)}")
    stat["L_words_added"] = len(L) - n0
    return stat


def chains(items, L, choice, GR, maxd=MAX_DEPTH):
    """問題ごとの点（種類, 語, 番号, 段の数）とつながり。分析は資料（g・s）を選んで鎖を作り直す。"""
    out = {}
    for it in items:
        dep, links = follow(L, choice, GR, it["ent"], maxd)
        out[it["id"]] = ([[t, w, k, d] for (t, w, k), d in sorted(dep.items())],
                         sorted([list(a), list(b)] for a, b in links))
    return out

In [ ]:
# 中身3（analysis/e15/e15_core.py と同じ）
"""E15 の中身（Colab と手元の両方で使う。手書きの語の一覧はありません）。
- 親の語（頭の名詞）：今まで＝NLI の段の値が一番の候補（"genus"）。E15＝Gemma に「説明文のどの語が、w が何の一種かを言っているか」を選ばせる（"ggenus"）。
- 最後の段（分類に着くか）：今まで＝その点の説明文1つで NLI（"w" means: 説明文 ⇒ W is a type of C.）。
  E15＝たどった道の説明文を全部見せて Gemma に「この説明文から、x は C の一種か」を聞く。
  対照：道の最後の説明文だけ（last）、関係のない別の問題の道（ctrl）、説明文なし（none）。
- 道の説明文に分類の語 C を足すことはしない（説明文そのものに C が書いてあるのは辞書の情報なのでそのまま）。
"""

MAXD = 3          # E8n と同じ3段
TOP_M = None      # Gemma に聞く点：全部（上から8つに絞ると、意味が8つある語で根の点だけで埋まり、E8n と 89問ずれた）


def gkey(mode):
    return "genus" if mode == "n" else "ggenus"


def step_from(L, CH, ST, mode, w, k):
    """(w,k) から親の点 (g,j) と段の強さ。無ければ None。"""
    g = L[w][k].get(gkey(mode))
    if not g or g not in L or g == w or not L[g]:
        return None
    j = CH[mode].get((w, k), 0) if len(L[g]) > 1 else 0
    return (g, j), ST[mode].get((w, k), (0.0, 0.0))[0]


def reach(L, CH, ST, mode, x, maxd=MAXD):
    """物の語 x の全部の意味から maxd 段まで。点 → (道の強さの掛け算, 道[点...])。同じ点に着く道は強い方。"""
    best = {}
    front = []
    for k in range(len(L.get(x, []))):
        best[(x, k)] = (1.0, [(x, k)]); front.append((x, k))
    for _ in range(maxd):
        nxt = []
        for n in front:
            s = step_from(L, CH, ST, mode, *n)
            if s is None:
                continue
            m, e = s
            p = best[n][0] * e
            if m not in best or p > best[m][0]:
                best[m] = (p, best[n][1] + [m]); nxt.append(m)
        front = nxt
    return best


def top_nodes(best, m=TOP_M):
    o = sorted(best, key=lambda n: (-best[n][0], len(best[n][1]), n))
    return o if m is None else o[:m]


def path_lines(L, path):
    return [f'- "{w}": {L[w][k]["def"]}' for w, k in path]


def arrive_prompt(x, C, lines):
    """lines が空なら説明文なしで聞く。"""
    if lines:
        return ("Dictionary definitions:\n" + "\n".join(lines) +
                f"\n\nBased on these definitions, is a \"{x}\" a kind of \"{C}\"? Answer Yes or No only.")
    return f"Is a \"{x}\" a kind of \"{C}\"? Answer Yes or No only."


def genus_prompt(w, d, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Dictionary definition of \"{w}\": {d}\n\nWhich of these words from the definition names the kind of thing "
            f"that a \"{w}\" is?\n{opts}\n\nAnswer with the number only.")


def score(best, arr, nodes):
    """道の強さ × 着き の一番大きい物。arr[n] が無い点は 0。"""
    v = [best[n][0] * arr.get(n, 0.0) for n in nodes if n in best]
    return max(v) if v else 0.0

In [ ]:
OLD = json.loads(r'''[{"id": 0, "ent": "barbell", "catw": "tool"}, {"id": 1, "ent": "tenement", "catw": "building"}, {"id": 2, "ent": "porpoise", "catw": "dog"}, {"id": 3, "ent": "juvenile", "catw": "boat"}, {"id": 4, "ent": "barbell", "catw": "mammal"}, {"id": 5, "ent": "grocer", "catw": "living thing"}, {"id": 6, "ent": "gramophone", "catw": "game"}, {"id": 7, "ent": "hooch", "catw": "artist"}, {"id": 8, "ent": "bilingual", "catw": "person"}, {"id": 9, "ent": "minnow", "catw": "disease"}, {"id": 10, "ent": "gazelle", "catw": "food"}, {"id": 11, "ent": "gasket", "catw": "game"}, {"id": 12, "ent": "isolationist", "catw": "living thing"}, {"id": 13, "ent": "clamp", "catw": "tool"}, {"id": 14, "ent": "copycat", "catw": "body part"}, {"id": 15, "ent": "peshmerga", "catw": "living thing"}, {"id": 16, "ent": "tabor", "catw": "boat"}, {"id": 17, "ent": "stepmother", "catw": "person"}, {"id": 18, "ent": "terrier", "catw": "bird"}, {"id": 19, "ent": "preteen", "catw": "metal"}, {"id": 20, "ent": "composer", "catw": "person"}, {"id": 21, "ent": "suitor", "catw": "living thing"}, {"id": 22, "ent": "leaker", "catw": "person"}, {"id": 23, "ent": "actuator", "catw": "machine"}, {"id": 24, "ent": "lally", "catw": "tool"}, {"id": 25, "ent": "epileptic", "catw": "disease"}, {"id": 26, "ent": "pew", "catw": "man-made object"}, {"id": 27, "ent": "microphone", "catw": "equipment"}, {"id": 28, "ent": "felt", "catw": "material"}, {"id": 29, "ent": "primrose", "catw": "living thing"}, {"id": 30, "ent": "quartermaster", "catw": "person"}, {"id": 31, "ent": "delegate", "catw": "person"}, {"id": 32, "ent": "grubby", "catw": "toy"}, {"id": 33, "ent": "prince", "catw": "artist"}, {"id": 34, "ent": "locomotive", "catw": "man-made object"}, {"id": 35, "ent": "lobbyist", "catw": "living thing"}, {"id": 36, "ent": "workman", "catw": "footwear"}, {"id": 37, "ent": "stupid", "catw": "person"}, {"id": 38, "ent": "watchman", "catw": "body part"}, {"id": 39, "ent": "headliner", "catw": "equipment"}, {"id": 40, "ent": "geranium", "catw": "living thing"}, {"id": 41, "ent": "guerrilla", "catw": "weapon"}, {"id": 42, "ent": "sheeting", "catw": "food"}, {"id": 43, "ent": "consulate", "catw": "device"}, {"id": 44, "ent": "dinky", "catw": "artist"}, {"id": 45, "ent": "caterer", "catw": "sport"}, {"id": 46, "ent": "rickey", "catw": "tree"}, {"id": 47, "ent": "headmistress", "catw": "worker"}, {"id": 48, "ent": "educator", "catw": "worker"}, {"id": 49, "ent": "golf", "catw": "metal"}, {"id": 50, "ent": "velvet", "catw": "man-made object"}, {"id": 51, "ent": "biographer", "catw": "disease"}, {"id": 52, "ent": "zoologist", "catw": "living thing"}, {"id": 53, "ent": "doge", "catw": "living thing"}, {"id": 54, "ent": "mimic", "catw": "reptile"}, {"id": 55, "ent": "passerby", "catw": "organ"}, {"id": 56, "ent": "dieter", "catw": "fish"}, {"id": 57, "ent": "niqab", "catw": "reptile"}, {"id": 58, "ent": "bulldozer", "catw": "machine"}, {"id": 59, "ent": "deflector", "catw": "equipment"}, {"id": 60, "ent": "schoolgirl", "catw": "metal"}, {"id": 61, "ent": "falconer", "catw": "sport"}, {"id": 62, "ent": "marlin", "catw": "drink"}, {"id": 63, "ent": "plumber", "catw": "bird"}, {"id": 64, "ent": "sarcophagus", "catw": "man-made object"}, {"id": 65, "ent": "amputee", "catw": "person"}, {"id": 66, "ent": "kimono", "catw": "fish"}, {"id": 67, "ent": "temp", "catw": "worker"}, {"id": 68, "ent": "propeller", "catw": "reptile"}, {"id": 69, "ent": "wombat", "catw": "building"}, {"id": 70, "ent": "paintbrush", "catw": "tool"}, {"id": 71, "ent": "womanizer", "catw": "body part"}, {"id": 72, "ent": "greenhouse", "catw": "man-made object"}, {"id": 73, "ent": "tactician", "catw": "food"}, {"id": 74, "ent": "achiever", "catw": "fruit"}, {"id": 75, "ent": "moll", "catw": "person"}, {"id": 76, "ent": "bonsai", "catw": "plant"}, {"id": 77, "ent": "underdog", "catw": "tree"}, {"id": 78, "ent": "loom", "catw": "car"}, {"id": 79, "ent": "individualist", "catw": "person"}, {"id": 80, "ent": "easel", "catw": "sport"}, {"id": 81, "ent": "finalist", "catw": "athlete"}, {"id": 82, "ent": "sagebrush", "catw": "weapon"}, {"id": 83, "ent": "uniform", "catw": "equipment"}, {"id": 84, "ent": "bonsai", "catw": "animal"}, {"id": 85, "ent": "receptionist", "catw": "worker"}, {"id": 86, "ent": "commoner", "catw": "worker"}, {"id": 87, "ent": "modernist", "catw": "person"}, {"id": 88, "ent": "cuppa", "catw": "container"}, {"id": 89, "ent": "cornerback", "catw": "athlete"}, {"id": 90, "ent": "minnow", "catw": "game"}, {"id": 91, "ent": "neurologist", "catw": "worker"}, {"id": 92, "ent": "kitten", "catw": "mammal"}, {"id": 93, "ent": "undercarriage", "catw": "device"}, {"id": 94, "ent": "reporter", "catw": "worker"}, {"id": 95, "ent": "critter", "catw": "equipment"}, {"id": 96, "ent": "granddaughter", "catw": "mammal"}, {"id": 97, "ent": "crusher", "catw": "machine"}, {"id": 98, "ent": "accomplice", "catw": "athlete"}, {"id": 99, "ent": "dubbing", "catw": "device"}, {"id": 100, "ent": "tortoise", "catw": "reptile"}, {"id": 101, "ent": "jukebox", "catw": "equipment"}, {"id": 102, "ent": "restaurant", "catw": "building"}, {"id": 103, "ent": "sectional", "catw": "man-made object"}, {"id": 104, "ent": "heifer", "catw": "living thing"}, {"id": 105, "ent": "kea", "catw": "living thing"}, {"id": 106, "ent": "minion", "catw": "living thing"}, {"id": 107, "ent": "opportunist", "catw": "person"}, {"id": 108, "ent": "conformist", "catw": "artist"}, {"id": 109, "ent": "burglar", "catw": "body part"}, {"id": 110, "ent": "buddy", "catw": "man-made object"}, {"id": 111, "ent": "wraparound", "catw": "metal"}, {"id": 112, "ent": "vise", "catw": "tool"}, {"id": 113, "ent": "helicopter", "catw": "man-made object"}, {"id": 114, "ent": "calender", "catw": "tool"}, {"id": 115, "ent": "snob", "catw": "game"}, {"id": 116, "ent": "drunkard", "catw": "living thing"}, {"id": 117, "ent": "bandicoot", "catw": "disease"}, {"id": 118, "ent": "musket", "catw": "vehicle"}, {"id": 119, "ent": "grader", "catw": "sport"}, {"id": 120, "ent": "briefcase", "catw": "vegetable"}, {"id": 121, "ent": "alternator", "catw": "vehicle"}, {"id": 122, "ent": "wallaby", "catw": "plant"}, {"id": 123, "ent": "interceptor", "catw": "aircraft"}, {"id": 124, "ent": "carbine", "catw": "equipment"}, {"id": 125, "ent": "spitfire", "catw": "living thing"}, {"id": 126, "ent": "huntress", "catw": "worker"}, {"id": 127, "ent": "heathen", "catw": "flower"}, {"id": 128, "ent": "senna", "catw": "man-made object"}, {"id": 129, "ent": "macho", "catw": "animal"}, {"id": 130, "ent": "beloved", "catw": "insect"}, {"id": 131, "ent": "wallaby", "catw": "living thing"}, {"id": 132, "ent": "accountant", "catw": "game"}, {"id": 133, "ent": "shogun", "catw": "fabric"}, {"id": 134, "ent": "parsonage", "catw": "boat"}, {"id": 135, "ent": "timepiece", "catw": "tool"}, {"id": 136, "ent": "biter", "catw": "living thing"}, {"id": 137, "ent": "rugby", "catw": "equipment"}, {"id": 138, "ent": "bedroom", "catw": "building"}, {"id": 139, "ent": "wombat", "catw": "tree"}, {"id": 140, "ent": "everlasting", "catw": "game"}, {"id": 141, "ent": "auctioneer", "catw": "material"}, {"id": 142, "ent": "theorist", "catw": "person"}, {"id": 143, "ent": "skater", "catw": "person"}, {"id": 144, "ent": "mariner", "catw": "metal"}, {"id": 145, "ent": "freelancer", "catw": "fish"}, {"id": 146, "ent": "dubbing", "catw": "game"}, {"id": 147, "ent": "simulator", "catw": "game"}, {"id": 148, "ent": "circuitry", "catw": "sport"}, {"id": 149, "ent": "galleon", "catw": "boat"}, {"id": 150, "ent": "steed", "catw": "animal"}, {"id": 151, "ent": "mosquito", "catw": "weapon"}, {"id": 152, "ent": "legate", "catw": "game"}, {"id": 153, "ent": "maxi", "catw": "room"}, {"id": 154, "ent": "bodybuilder", "catw": "animal"}, {"id": 155, "ent": "newlywed", "catw": "living thing"}, {"id": 156, "ent": "finch", "catw": "material"}, {"id": 157, "ent": "flea", "catw": "vegetable"}, {"id": 158, "ent": "betel", "catw": "tree"}, {"id": 159, "ent": "microphone", "catw": "sport"}, {"id": 160, "ent": "airliner", "catw": "boat"}, {"id": 161, "ent": "commodore", "catw": "machine"}, {"id": 162, "ent": "billionaire", "catw": "body part"}, {"id": 163, "ent": "roadhouse", "catw": "game"}, {"id": 164, "ent": "hater", "catw": "person"}, {"id": 165, "ent": "marc", "catw": "fruit"}, {"id": 166, "ent": "tugboat", "catw": "disease"}, {"id": 167, "ent": "blouse", "catw": "fish"}, {"id": 168, "ent": "houseboat", "catw": "device"}, {"id": 169, "ent": "bovine", "catw": "toy"}, {"id": 170, "ent": "porpoise", "catw": "athlete"}, {"id": 171, "ent": "everlasting", "catw": "plant"}, {"id": 172, "ent": "tomahawk", "catw": "man-made object"}, {"id": 173, "ent": "national", "catw": "boat"}, {"id": 174, "ent": "underpants", "catw": "man-made object"}, {"id": 175, "ent": "hitchhiker", "catw": "worker"}, {"id": 176, "ent": "cutthroat", "catw": "plant"}, {"id": 177, "ent": "theologian", "catw": "reptile"}, {"id": 178, "ent": "facilitator", "catw": "vegetable"}, {"id": 179, "ent": "rickey", "catw": "man-made object"}, {"id": 180, "ent": "songwriter", "catw": "dish"}, {"id": 181, "ent": "extremist", "catw": "fruit"}, {"id": 182, "ent": "transvestite", "catw": "sport"}, {"id": 183, "ent": "warship", "catw": "fruit"}, {"id": 184, "ent": "spokeswoman", "catw": "body part"}, {"id": 185, "ent": "brill", "catw": "animal"}, {"id": 186, "ent": "secessionist", "catw": "metal"}, {"id": 187, "ent": "beloved", "catw": "fabric"}, {"id": 188, "ent": "impala", "catw": "sport"}, {"id": 189, "ent": "firearm", "catw": "man-made object"}, {"id": 190, "ent": "mathematician", "catw": "worker"}, {"id": 191, "ent": "buttercup", "catw": "man-made object"}, {"id": 192, "ent": "notary", "catw": "game"}, {"id": 193, "ent": "missus", "catw": "boat"}, {"id": 194, "ent": "freelancer", "catw": "living thing"}, {"id": 195, "ent": "beret", "catw": "toy"}, {"id": 196, "ent": "georgette", "catw": "clothing"}, {"id": 197, "ent": "parsonage", "catw": "man-made object"}, {"id": 198, "ent": "dongle", "catw": "metal"}, {"id": 199, "ent": "spacecraft", "catw": "vehicle"}, {"id": 200, "ent": "eunuch", "catw": "living thing"}, {"id": 201, "ent": "apparel", "catw": "clothing"}, {"id": 202, "ent": "messenger", "catw": "person"}, {"id": 203, "ent": "strainer", "catw": "game"}, {"id": 204, "ent": "bookkeeper", "catw": "living thing"}, {"id": 205, "ent": "clamp", "catw": "musical instrument"}, {"id": 206, "ent": "sieve", "catw": "body part"}, {"id": 207, "ent": "madame", "catw": "insect"}, {"id": 208, "ent": "bethel", "catw": "body part"}, {"id": 209, "ent": "moray", "catw": "disease"}, {"id": 210, "ent": "lemur", "catw": "animal"}, {"id": 211, "ent": "orangutan", "catw": "fruit"}, {"id": 212, "ent": "athlete", "catw": "living thing"}, {"id": 213, "ent": "vole", "catw": "worker"}, {"id": 214, "ent": "peony", "catw": "car"}, {"id": 215, "ent": "undercarriage", "catw": "man-made object"}, {"id": 216, "ent": "blacksmith", "catw": "artist"}, {"id": 217, "ent": "erica", "catw": "game"}, {"id": 218, "ent": "clinician", "catw": "person"}, {"id": 219, "ent": "beekeeper", "catw": "person"}, {"id": 220, "ent": "purveyor", "catw": "living thing"}, {"id": 221, "ent": "carbine", "catw": "man-made object"}, {"id": 222, "ent": "ani", "catw": "animal"}, {"id": 223, "ent": "buyer", "catw": "material"}, {"id": 224, "ent": "bankrupt", "catw": "man-made object"}, {"id": 225, "ent": "landlady", "catw": "flower"}, {"id": 226, "ent": "winger", "catw": "aircraft"}, {"id": 227, "ent": "meatball", "catw": "food"}, {"id": 228, "ent": "autofocus", "catw": "man-made object"}, {"id": 229, "ent": "narrator", "catw": "clothing"}, {"id": 230, "ent": "winch", "catw": "musical instrument"}, {"id": 231, "ent": "hoist", "catw": "car"}, {"id": 232, "ent": "poodle", "catw": "disease"}, {"id": 233, "ent": "hamster", "catw": "athlete"}, {"id": 234, "ent": "finch", "catw": "bird"}, {"id": 235, "ent": "shrine", "catw": "vehicle"}, {"id": 236, "ent": "ivy", "catw": "car"}, {"id": 237, "ent": "pensioner", "catw": "person"}, {"id": 238, "ent": "flatmate", "catw": "organ"}, {"id": 239, "ent": "communicator", "catw": "living thing"}, {"id": 240, "ent": "menial", "catw": "plant"}, {"id": 241, "ent": "grandparent", "catw": "food"}, {"id": 242, "ent": "chalet", "catw": "building"}, {"id": 243, "ent": "avenger", "catw": "vegetable"}, {"id": 244, "ent": "gazebo", "catw": "fish"}, {"id": 245, "ent": "marten", "catw": "living thing"}, {"id": 246, "ent": "uterus", "catw": "container"}, {"id": 247, "ent": "mockingbird", "catw": "person"}, {"id": 248, "ent": "milkman", "catw": "living thing"}, {"id": 249, "ent": "sauna", "catw": "animal"}, {"id": 250, "ent": "aviator", "catw": "man-made object"}, {"id": 251, "ent": "freeman", "catw": "worker"}, {"id": 252, "ent": "salesman", "catw": "fish"}, {"id": 253, "ent": "synchrotron", "catw": "device"}, {"id": 254, "ent": "traveler", "catw": "reptile"}, {"id": 255, "ent": "skewer", "catw": "game"}, {"id": 256, "ent": "sleuth", "catw": "metal"}, {"id": 257, "ent": "abolitionist", "catw": "artist"}, {"id": 258, "ent": "mantel", "catw": "device"}, {"id": 259, "ent": "consulate", "catw": "building"}, {"id": 260, "ent": "minster", "catw": "boat"}, {"id": 261, "ent": "informer", "catw": "fruit"}, {"id": 262, "ent": "waitress", "catw": "food"}, {"id": 263, "ent": "loudspeaker", "catw": "man-made object"}, {"id": 264, "ent": "orator", "catw": "footwear"}, {"id": 265, "ent": "drummer", "catw": "worker"}, {"id": 266, "ent": "dingo", "catw": "animal"}, {"id": 267, "ent": "sailboat", "catw": "machine"}, {"id": 268, "ent": "idealist", "catw": "disease"}, {"id": 269, "ent": "cactus", "catw": "weapon"}, {"id": 270, "ent": "basset", "catw": "animal"}, {"id": 271, "ent": "handkerchief", "catw": "man-made object"}, {"id": 272, "ent": "plumber", "catw": "tree"}, {"id": 273, "ent": "webcam", "catw": "man-made object"}, {"id": 274, "ent": "dinghy", "catw": "boat"}, {"id": 275, "ent": "mod", "catw": "metal"}, {"id": 276, "ent": "erotic", "catw": "flower"}, {"id": 277, "ent": "songbird", "catw": "room"}, {"id": 278, "ent": "sauerkraut", "catw": "fruit"}, {"id": 279, "ent": "plenipotentiary", "catw": "equipment"}, {"id": 280, "ent": "dispensary", "catw": "man-made object"}, {"id": 281, "ent": "fetus", "catw": "man-made object"}, {"id": 282, "ent": "orchid", "catw": "artist"}, {"id": 283, "ent": "basset", "catw": "dog"}, {"id": 284, "ent": "coordinator", "catw": "person"}, {"id": 285, "ent": "taster", "catw": "worker"}, {"id": 286, "ent": "swot", "catw": "sport"}, {"id": 287, "ent": "exporter", "catw": "material"}, {"id": 288, "ent": "underwear", "catw": "clothing"}, {"id": 289, "ent": "pendulum", "catw": "device"}, {"id": 290, "ent": "stoner", "catw": "bird"}, {"id": 291, "ent": "bartender", "catw": "game"}, {"id": 292, "ent": "juggler", "catw": "living thing"}, {"id": 293, "ent": "sovereign", "catw": "game"}, {"id": 294, "ent": "stepson", "catw": "mammal"}, {"id": 295, "ent": "mongoose", "catw": "living thing"}, {"id": 296, "ent": "hater", "catw": "dog"}, {"id": 297, "ent": "cutthroat", "catw": "game"}, {"id": 298, "ent": "pedophile", "catw": "food"}, {"id": 299, "ent": "ripper", "catw": "plant"}, {"id": 300, "ent": "sultan", "catw": "organ"}, {"id": 301, "ent": "assistant", "catw": "dish"}, {"id": 302, "ent": "coauthor", "catw": "vegetable"}, {"id": 303, "ent": "ladybug", "catw": "equipment"}, {"id": 304, "ent": "dredge", "catw": "man-made object"}, {"id": 305, "ent": "clarence", "catw": "vehicle"}, {"id": 306, "ent": "eft", "catw": "aircraft"}, {"id": 307, "ent": "modernist", "catw": "building"}, {"id": 308, "ent": "satchel", "catw": "man-made object"}, {"id": 309, "ent": "invalid", "catw": "game"}, {"id": 310, "ent": "jogger", "catw": "food"}, {"id": 311, "ent": "rancher", "catw": "body part"}, {"id": 312, "ent": "clary", "catw": "body part"}, {"id": 313, "ent": "recliner", "catw": "body part"}, {"id": 314, "ent": "mercenary", "catw": "worker"}, {"id": 315, "ent": "storeroom", "catw": "vegetable"}, {"id": 316, "ent": "protege", "catw": "vegetable"}, {"id": 317, "ent": "handyman", "catw": "weapon"}, {"id": 318, "ent": "wanton", "catw": "living thing"}, {"id": 319, "ent": "bloodhound", "catw": "vegetable"}, {"id": 320, "ent": "crustacean", "catw": "body part"}, {"id": 321, "ent": "electric", "catw": "car"}, {"id": 322, "ent": "itinerant", "catw": "artist"}, {"id": 323, "ent": "outhouse", "catw": "container"}, {"id": 324, "ent": "searchlight", "catw": "metal"}, {"id": 325, "ent": "stickler", "catw": "person"}, {"id": 326, "ent": "snob", "catw": "person"}, {"id": 327, "ent": "erotic", "catw": "living thing"}, {"id": 328, "ent": "violin", "catw": "musical instrument"}, {"id": 329, "ent": "plover", "catw": "bird"}, {"id": 330, "ent": "lobbyist", "catw": "boat"}, {"id": 331, "ent": "motorcyclist", "catw": "footwear"}, {"id": 332, "ent": "harlequin", "catw": "reptile"}, {"id": 333, "ent": "cheeseburger", "catw": "man-made object"}, {"id": 334, "ent": "wearer", "catw": "mammal"}, {"id": 335, "ent": "geisha", "catw": "plant"}, {"id": 336, "ent": "poppy", "catw": "sport"}, {"id": 337, "ent": "bobbin", "catw": "fruit"}, {"id": 338, "ent": "preacher", "catw": "person"}, {"id": 339, "ent": "ladybug", "catw": "insect"}, {"id": 340, "ent": "cubby", "catw": "living thing"}, {"id": 341, "ent": "neurotic", "catw": "metal"}, {"id": 342, "ent": "applicator", "catw": "device"}, {"id": 343, "ent": "hoarder", "catw": "mammal"}, {"id": 344, "ent": "escapist", "catw": "game"}, {"id": 345, "ent": "bereaved", "catw": "living thing"}, {"id": 346, "ent": "termite", "catw": "mammal"}, {"id": 347, "ent": "dragonfly", "catw": "fish"}, {"id": 348, "ent": "razor", "catw": "weapon"}, {"id": 349, "ent": "mensch", "catw": "person"}, {"id": 350, "ent": "kingfisher", "catw": "insect"}, {"id": 351, "ent": "snob", "catw": "vegetable"}, {"id": 352, "ent": "bock", "catw": "vegetable"}, {"id": 353, "ent": "accelerometer", "catw": "sport"}, {"id": 354, "ent": "electrode", "catw": "equipment"}, {"id": 355, "ent": "subcontractor", "catw": "food"}, {"id": 356, "ent": "vase", "catw": "container"}, {"id": 357, "ent": "parachute", "catw": "equipment"}, {"id": 358, "ent": "soloist", "catw": "metal"}, {"id": 359, "ent": "prankster", "catw": "person"}, {"id": 360, "ent": "missus", "catw": "person"}, {"id": 361, "ent": "gynecologist", "catw": "living thing"}, {"id": 362, "ent": "landowner", "catw": "person"}, {"id": 363, "ent": "mosquito", "catw": "mammal"}, {"id": 364, "ent": "decadent", "catw": "dish"}, {"id": 365, "ent": "toolbox", "catw": "tool"}, {"id": 366, "ent": "brandy", "catw": "food"}, {"id": 367, "ent": "housekeeper", "catw": "flower"}, {"id": 368, "ent": "convener", "catw": "material"}, {"id": 369, "ent": "moa", "catw": "animal"}, {"id": 370, "ent": "advertiser", "catw": "metal"}, {"id": 371, "ent": "spaniel", "catw": "dog"}, {"id": 372, "ent": "denture", "catw": "man-made object"}, {"id": 373, "ent": "pigeon", "catw": "person"}, {"id": 374, "ent": "chrysalis", "catw": "body part"}, {"id": 375, "ent": "vizier", "catw": "athlete"}, {"id": 376, "ent": "bandit", "catw": "device"}, {"id": 377, "ent": "dominatrix", "catw": "living thing"}, {"id": 378, "ent": "protectionist", "catw": "plant"}, {"id": 379, "ent": "cyborg", "catw": "body part"}, {"id": 380, "ent": "manger", "catw": "worker"}, {"id": 381, "ent": "stowaway", "catw": "vegetable"}, {"id": 382, "ent": "stingray", "catw": "living thing"}, {"id": 383, "ent": "immigrant", "catw": "game"}, {"id": 384, "ent": "peripheral", "catw": "animal"}, {"id": 385, "ent": "corgi", "catw": "clothing"}, {"id": 386, "ent": "trusty", "catw": "game"}, {"id": 387, "ent": "inventor", "catw": "drink"}, {"id": 388, "ent": "narc", "catw": "person"}, {"id": 389, "ent": "brazier", "catw": "drink"}, {"id": 390, "ent": "curate", "catw": "living thing"}, {"id": 391, "ent": "stagecoach", "catw": "vehicle"}, {"id": 392, "ent": "molester", "catw": "mammal"}, {"id": 393, "ent": "paramedic", "catw": "worker"}, {"id": 394, "ent": "inventor", "catw": "body part"}, {"id": 395, "ent": "synagogue", "catw": "vegetable"}, {"id": 396, "ent": "adolescent", "catw": "mammal"}, {"id": 397, "ent": "contrarian", "catw": "metal"}, {"id": 398, "ent": "swimsuit", "catw": "equipment"}, {"id": 399, "ent": "shrine", "catw": "container"}, {"id": 400, "ent": "manatee", "catw": "vehicle"}, {"id": 401, "ent": "toddler", "catw": "tool"}, {"id": 402, "ent": "introvert", "catw": "worker"}, {"id": 403, "ent": "copywriter", "catw": "clothing"}, {"id": 404, "ent": "vigilante", "catw": "person"}, {"id": 405, "ent": "cuttlefish", "catw": "animal"}, {"id": 406, "ent": "alderman", "catw": "athlete"}, {"id": 407, "ent": "hater", "catw": "boat"}, {"id": 408, "ent": "airship", "catw": "dog"}, {"id": 409, "ent": "benefactor", "catw": "person"}, {"id": 410, "ent": "financier", "catw": "worker"}, {"id": 411, "ent": "trader", "catw": "worker"}, {"id": 412, "ent": "anti", "catw": "living thing"}, {"id": 413, "ent": "sarcophagus", "catw": "tool"}, {"id": 414, "ent": "paleontologist", "catw": "worker"}, {"id": 415, "ent": "accordion", "catw": "fish"}, {"id": 416, "ent": "sphincter", "catw": "disease"}, {"id": 417, "ent": "detainee", "catw": "living thing"}, {"id": 418, "ent": "pleaser", "catw": "game"}, {"id": 419, "ent": "princess", "catw": "person"}, {"id": 420, "ent": "paramedic", "catw": "dish"}, {"id": 421, "ent": "rink", "catw": "sport"}, {"id": 422, "ent": "propagandist", "catw": "person"}, {"id": 423, "ent": "confidante", "catw": "person"}, {"id": 424, "ent": "stockbroker", "catw": "footwear"}, {"id": 425, "ent": "drosophila", "catw": "food"}, {"id": 426, "ent": "pacifist", "catw": "insect"}, {"id": 427, "ent": "suitor", "catw": "game"}, {"id": 428, "ent": "apologist", "catw": "worker"}, {"id": 429, "ent": "clubhouse", "catw": "room"}, {"id": 430, "ent": "strategist", "catw": "person"}, {"id": 431, "ent": "jeep", "catw": "container"}, {"id": 432, "ent": "survivalist", "catw": "fish"}, {"id": 433, "ent": "vermouth", "catw": "plant"}, {"id": 434, "ent": "contrarian", "catw": "animal"}, {"id": 435, "ent": "cattle", "catw": "animal"}, {"id": 436, "ent": "pensioner", "catw": "sport"}, {"id": 437, "ent": "marlin", "catw": "living thing"}, {"id": 438, "ent": "mockingbird", "catw": "bird"}, {"id": 439, "ent": "dandelion", "catw": "mammal"}, {"id": 440, "ent": "meatball", "catw": "game"}, {"id": 441, "ent": "chime", "catw": "organ"}, {"id": 442, "ent": "hornet", "catw": "worker"}, {"id": 443, "ent": "wildflower", "catw": "living thing"}, {"id": 444, "ent": "gyroscope", "catw": "machine"}, {"id": 445, "ent": "lender", "catw": "bird"}, {"id": 446, "ent": "kangaroo", "catw": "athlete"}, {"id": 447, "ent": "calico", "catw": "fabric"}, {"id": 448, "ent": "tongs", "catw": "tool"}, {"id": 449, "ent": "cadet", "catw": "dish"}, {"id": 450, "ent": "repairman", "catw": "fruit"}, {"id": 451, "ent": "clubhouse", "catw": "machine"}, {"id": 452, "ent": "hummingbird", "catw": "animal"}, {"id": 453, "ent": "bison", "catw": "mammal"}, {"id": 454, "ent": "piglet", "catw": "mammal"}, {"id": 455, "ent": "hyena", "catw": "plant"}, {"id": 456, "ent": "wife", "catw": "mammal"}, {"id": 457, "ent": "berk", "catw": "living thing"}, {"id": 458, "ent": "transistor", "catw": "fish"}, {"id": 459, "ent": "firstborn", "catw": "sport"}, {"id": 460, "ent": "interviewee", "catw": "worker"}, {"id": 461, "ent": "surrealist", "catw": "fish"}, {"id": 462, "ent": "corrective", "catw": "man-made object"}, {"id": 463, "ent": "searchlight", "catw": "equipment"}, {"id": 464, "ent": "bookkeeper", "catw": "reptile"}, {"id": 465, "ent": "takin", "catw": "fish"}, {"id": 466, "ent": "therapist", "catw": "living thing"}, {"id": 467, "ent": "communicator", "catw": "person"}, {"id": 468, "ent": "bereaved", "catw": "game"}, {"id": 469, "ent": "apparel", "catw": "weapon"}, {"id": 470, "ent": "boardroom", "catw": "room"}, {"id": 471, "ent": "goldilocks", "catw": "flower"}, {"id": 472, "ent": "manatee", "catw": "food"}, {"id": 473, "ent": "aloe", "catw": "room"}, {"id": 474, "ent": "congressman", "catw": "flower"}, {"id": 475, "ent": "ecologist", "catw": "worker"}, {"id": 476, "ent": "dahlia", "catw": "plant"}, {"id": 477, "ent": "lodger", "catw": "plant"}, {"id": 478, "ent": "flea", "catw": "animal"}, {"id": 479, "ent": "hillbilly", "catw": "living thing"}, {"id": 480, "ent": "stewardess", "catw": "equipment"}, {"id": 481, "ent": "helmsman", "catw": "athlete"}, {"id": 482, "ent": "jasmine", "catw": "vegetable"}, {"id": 483, "ent": "palmetto", "catw": "musical instrument"}, {"id": 484, "ent": "rapporteur", "catw": "organ"}, {"id": 485, "ent": "bobcat", "catw": "mammal"}, {"id": 486, "ent": "cutler", "catw": "living thing"}, {"id": 487, "ent": "hearse", "catw": "man-made object"}, {"id": 488, "ent": "spastic", "catw": "reptile"}, {"id": 489, "ent": "alcoholic", "catw": "drink"}, {"id": 490, "ent": "longhorn", "catw": "tree"}, {"id": 491, "ent": "beagle", "catw": "sport"}, {"id": 492, "ent": "impressionist", "catw": "fabric"}, {"id": 493, "ent": "inpatient", "catw": "fruit"}, {"id": 494, "ent": "emir", "catw": "boat"}, {"id": 495, "ent": "snapdragon", "catw": "living thing"}, {"id": 496, "ent": "stockholder", "catw": "furniture"}, {"id": 497, "ent": "schizophrenic", "catw": "sport"}, {"id": 498, "ent": "rapporteur", "catw": "person"}, {"id": 499, "ent": "ombudsman", "catw": "living thing"}, {"id": 500, "ent": "cleric", "catw": "plant"}, {"id": 501, "ent": "patriot", "catw": "aircraft"}, {"id": 502, "ent": "homophobe", "catw": "sport"}, {"id": 503, "ent": "rifle", "catw": "animal"}, {"id": 504, "ent": "phytoplankton", "catw": "plant"}, {"id": 505, "ent": "caveman", "catw": "person"}, {"id": 506, "ent": "exporter", "catw": "drink"}, {"id": 507, "ent": "sibling", "catw": "mammal"}, {"id": 508, "ent": "limousine", "catw": "clothing"}, {"id": 509, "ent": "lyre", "catw": "man-made object"}, {"id": 510, "ent": "audiovisual", "catw": "material"}, {"id": 511, "ent": "communicator", "catw": "sport"}, {"id": 512, "ent": "bathrobe", "catw": "metal"}, {"id": 513, "ent": "thwart", "catw": "device"}, {"id": 514, "ent": "stork", "catw": "aircraft"}, {"id": 515, "ent": "marksman", "catw": "game"}, {"id": 516, "ent": "confessional", "catw": "toy"}, {"id": 517, "ent": "monastery", "catw": "container"}, {"id": 518, "ent": "headdress", "catw": "flower"}, {"id": 519, "ent": "statesman", "catw": "living thing"}, {"id": 520, "ent": "milkman", "catw": "vegetable"}, {"id": 521, "ent": "thermometer", "catw": "tool"}, {"id": 522, "ent": "expressionist", "catw": "game"}, {"id": 523, "ent": "perpetrator", "catw": "insect"}, {"id": 524, "ent": "coupe", "catw": "container"}, {"id": 525, "ent": "councilman", "catw": "worker"}, {"id": 526, "ent": "authoritarian", "catw": "sport"}, {"id": 527, "ent": "satirist", "catw": "artist"}, {"id": 528, "ent": "antelope", "catw": "living thing"}, {"id": 529, "ent": "scraper", "catw": "device"}, {"id": 530, "ent": "ballroom", "catw": "sport"}, {"id": 531, "ent": "dunce", "catw": "flower"}, {"id": 532, "ent": "hiker", "catw": "athlete"}, {"id": 533, "ent": "playmate", "catw": "mammal"}, {"id": 534, "ent": "agitator", "catw": "vegetable"}, {"id": 535, "ent": "stiletto", "catw": "footwear"}, {"id": 536, "ent": "clock", "catw": "machine"}, {"id": 537, "ent": "bankrupt", "catw": "person"}, {"id": 538, "ent": "sophomore", "catw": "boat"}, {"id": 539, "ent": "hillbilly", "catw": "material"}, {"id": 540, "ent": "yarrow", "catw": "flower"}, {"id": 541, "ent": "caveman", "catw": "living thing"}, {"id": 542, "ent": "battleship", "catw": "person"}, {"id": 543, "ent": "ratchet", "catw": "tool"}, {"id": 544, "ent": "rival", "catw": "weapon"}, {"id": 545, "ent": "candlestick", "catw": "device"}, {"id": 546, "ent": "helmsman", "catw": "boat"}, {"id": 547, "ent": "suitor", "catw": "person"}, {"id": 548, "ent": "greenhouse", "catw": "device"}, {"id": 549, "ent": "socialist", "catw": "person"}, {"id": 550, "ent": "prince", "catw": "person"}, {"id": 551, "ent": "pulley", "catw": "footwear"}, {"id": 552, "ent": "parachute", "catw": "device"}, {"id": 553, "ent": "crocodile", "catw": "body part"}, {"id": 554, "ent": "innocent", "catw": "man-made object"}, {"id": 555, "ent": "conch", "catw": "living thing"}, {"id": 556, "ent": "electrode", "catw": "car"}, {"id": 557, "ent": "raglan", "catw": "furniture"}, {"id": 558, "ent": "nacho", "catw": "material"}, {"id": 559, "ent": "surgeon", "catw": "worker"}, {"id": 560, "ent": "artist", "catw": "fish"}, {"id": 561, "ent": "claimant", "catw": "disease"}, {"id": 562, "ent": "ibis", "catw": "footwear"}, {"id": 563, "ent": "notary", "catw": "person"}, {"id": 564, "ent": "motorboat", "catw": "machine"}, {"id": 565, "ent": "loom", "catw": "man-made object"}, {"id": 566, "ent": "firearm", "catw": "weapon"}, {"id": 567, "ent": "haggis", "catw": "organ"}, {"id": 568, "ent": "brogan", "catw": "food"}, {"id": 569, "ent": "tam", "catw": "dish"}, {"id": 570, "ent": "motorcyclist", "catw": "sport"}, {"id": 571, "ent": "alumnus", "catw": "plant"}, {"id": 572, "ent": "dildo", "catw": "equipment"}, {"id": 573, "ent": "tights", "catw": "vegetable"}, {"id": 574, "ent": "doorbell", "catw": "boat"}, {"id": 575, "ent": "letterman", "catw": "reptile"}, {"id": 576, "ent": "athlete", "catw": "worker"}, {"id": 577, "ent": "hostage", "catw": "game"}, {"id": 578, "ent": "empty", "catw": "container"}, {"id": 579, "ent": "spacecraft", "catw": "machine"}, {"id": 580, "ent": "kilt", "catw": "boat"}, {"id": 581, "ent": "shook", "catw": "body part"}, {"id": 582, "ent": "headscarf", "catw": "clothing"}, {"id": 583, "ent": "geranium", "catw": "animal"}, {"id": 584, "ent": "mangrove", "catw": "weapon"}, {"id": 585, "ent": "librarian", "catw": "food"}, {"id": 586, "ent": "restaurateur", "catw": "athlete"}, {"id": 587, "ent": "perfectionist", "catw": "living thing"}, {"id": 588, "ent": "flashlight", "catw": "tool"}, {"id": 589, "ent": "groundhog", "catw": "man-made object"}, {"id": 590, "ent": "contemplative", "catw": "footwear"}, {"id": 591, "ent": "robber", "catw": "living thing"}, {"id": 592, "ent": "tequila", "catw": "plant"}, {"id": 593, "ent": "expert", "catw": "person"}, {"id": 594, "ent": "wheelchair", "catw": "game"}, {"id": 595, "ent": "cuttlefish", "catw": "artist"}, {"id": 596, "ent": "traveler", "catw": "game"}, {"id": 597, "ent": "thermos", "catw": "plant"}, {"id": 598, "ent": "customer", "catw": "weapon"}, {"id": 599, "ent": "loner", "catw": "living thing"}, {"id": 600, "ent": "slipknot", "catw": "material"}, {"id": 601, "ent": "landlady", "catw": "game"}, {"id": 602, "ent": "oppressor", "catw": "device"}, {"id": 603, "ent": "alderman", "catw": "boat"}, {"id": 604, "ent": "fife", "catw": "fabric"}, {"id": 605, "ent": "loudspeaker", "catw": "disease"}, {"id": 606, "ent": "revisionist", "catw": "artist"}, {"id": 607, "ent": "plenipotentiary", "catw": "mammal"}, {"id": 608, "ent": "houseboat", "catw": "vehicle"}, {"id": 609, "ent": "optimist", "catw": "body part"}, {"id": 610, "ent": "primrose", "catw": "flower"}, {"id": 611, "ent": "moray", "catw": "insect"}, {"id": 612, "ent": "roebuck", "catw": "animal"}, {"id": 613, "ent": "cafe", "catw": "man-made object"}, {"id": 614, "ent": "proletarian", "catw": "worker"}, {"id": 615, "ent": "limiter", "catw": "man-made object"}, {"id": 616, "ent": "brogan", "catw": "man-made object"}, {"id": 617, "ent": "wingman", "catw": "game"}, {"id": 618, "ent": "robber", "catw": "game"}, {"id": 619, "ent": "aquarium", "catw": "boat"}, {"id": 620, "ent": "handkerchief", "catw": "device"}, {"id": 621, "ent": "legislator", "catw": "worker"}, {"id": 622, "ent": "preteen", "catw": "person"}, {"id": 623, "ent": "individualist", "catw": "worker"}, {"id": 624, "ent": "detonator", "catw": "furniture"}, {"id": 625, "ent": "banjo", "catw": "fish"}, {"id": 626, "ent": "neurosurgeon", "catw": "footwear"}, {"id": 627, "ent": "sunflower", "catw": "living thing"}, {"id": 628, "ent": "kylie", "catw": "aircraft"}, {"id": 629, "ent": "nawab", "catw": "clothing"}, {"id": 630, "ent": "assistant", "catw": "person"}, {"id": 631, "ent": "cantilever", "catw": "body part"}, {"id": 632, "ent": "libertine", "catw": "fish"}, {"id": 633, "ent": "chronicler", "catw": "artist"}, {"id": 634, "ent": "earl", "catw": "boat"}, {"id": 635, "ent": "abbess", "catw": "building"}, {"id": 636, "ent": "murine", "catw": "food"}, {"id": 637, "ent": "stole", "catw": "weapon"}, {"id": 638, "ent": "stapler", "catw": "boat"}, {"id": 639, "ent": "ski", "catw": "device"}, {"id": 640, "ent": "materiel", "catw": "equipment"}, {"id": 641, "ent": "rheumatic", "catw": "living thing"}, {"id": 642, "ent": "margarita", "catw": "dish"}, {"id": 643, "ent": "dredge", "catw": "animal"}, {"id": 644, "ent": "private", "catw": "fruit"}, {"id": 645, "ent": "chalet", "catw": "car"}, {"id": 646, "ent": "locket", "catw": "device"}, {"id": 647, "ent": "earl", "catw": "animal"}, {"id": 648, "ent": "blazer", "catw": "weapon"}, {"id": 649, "ent": "jammer", "catw": "body part"}, {"id": 650, "ent": "crustacean", "catw": "food"}, {"id": 651, "ent": "android", "catw": "worker"}, {"id": 652, "ent": "cowgirl", "catw": "animal"}, {"id": 653, "ent": "entrepreneur", "catw": "food"}, {"id": 654, "ent": "implant", "catw": "body part"}, {"id": 655, "ent": "camcorder", "catw": "device"}, {"id": 656, "ent": "macadamia", "catw": "living thing"}, {"id": 657, "ent": "waterfowl", "catw": "game"}, {"id": 658, "ent": "celibate", "catw": "person"}, {"id": 659, "ent": "rhinoceros", "catw": "person"}, {"id": 660, "ent": "blazer", "catw": "man-made object"}, {"id": 661, "ent": "retriever", "catw": "game"}, {"id": 662, "ent": "antelope", "catw": "footwear"}, {"id": 663, "ent": "wally", "catw": "sport"}, {"id": 664, "ent": "usurper", "catw": "game"}, {"id": 665, "ent": "cultivar", "catw": "material"}, {"id": 666, "ent": "chalet", "catw": "game"}, {"id": 667, "ent": "kitchen", "catw": "equipment"}, {"id": 668, "ent": "matchbox", "catw": "toy"}, {"id": 669, "ent": "homeowner", "catw": "person"}, {"id": 670, "ent": "candlestick", "catw": "tool"}, {"id": 671, "ent": "priestess", "catw": "dog"}, {"id": 672, "ent": "toddler", "catw": "mammal"}, {"id": 673, "ent": "detonator", "catw": "man-made object"}, {"id": 674, "ent": "ax", "catw": "man-made object"}, {"id": 675, "ent": "lingerie", "catw": "equipment"}, {"id": 676, "ent": "biochemist", "catw": "worker"}, {"id": 677, "ent": "clary", "catw": "weapon"}, {"id": 678, "ent": "juvenile", "catw": "person"}, {"id": 679, "ent": "psychologist", "catw": "worker"}, {"id": 680, "ent": "obstetrician", "catw": "person"}, {"id": 681, "ent": "biologist", "catw": "worker"}, {"id": 682, "ent": "camcorder", "catw": "room"}, {"id": 683, "ent": "aristocrat", "catw": "fruit"}, {"id": 684, "ent": "tripod", "catw": "equipment"}, {"id": 685, "ent": "ashtray", "catw": "metal"}, {"id": 686, "ent": "atheist", "catw": "dog"}, {"id": 687, "ent": "airship", "catw": "body part"}, {"id": 688, "ent": "greenhouse", "catw": "container"}, {"id": 689, "ent": "canoe", "catw": "vehicle"}, {"id": 690, "ent": "ship", "catw": "tool"}, {"id": 691, "ent": "jodhpur", "catw": "vehicle"}, {"id": 692, "ent": "coffin", "catw": "footwear"}, {"id": 693, "ent": "wrestler", "catw": "athlete"}, {"id": 694, "ent": "dominatrix", "catw": "vehicle"}, {"id": 695, "ent": "caribou", "catw": "game"}, {"id": 696, "ent": "harmonica", "catw": "equipment"}, {"id": 697, "ent": "genitalia", "catw": "vegetable"}, {"id": 698, "ent": "anesthesiologist", "catw": "worker"}, {"id": 699, "ent": "golf", "catw": "equipment"}]''')
NEW = json.loads(r'''[{"id": 1000, "ent": "racist", "catw": "plant"}, {"id": 1001, "ent": "goggles", "catw": "equipment"}, {"id": 1002, "ent": "swatch", "catw": "man-made object"}, {"id": 1003, "ent": "payer", "catw": "machine"}, {"id": 1004, "ent": "mater", "catw": "material"}, {"id": 1005, "ent": "fancier", "catw": "game"}, {"id": 1006, "ent": "electrician", "catw": "plant"}, {"id": 1007, "ent": "crucible", "catw": "man-made object"}, {"id": 1008, "ent": "consul", "catw": "tool"}, {"id": 1009, "ent": "dropper", "catw": "tool"}, {"id": 1010, "ent": "juror", "catw": "material"}, {"id": 1011, "ent": "abbot", "catw": "container"}, {"id": 1012, "ent": "viper", "catw": "weapon"}, {"id": 1013, "ent": "midge", "catw": "living thing"}, {"id": 1014, "ent": "godmother", "catw": "bird"}, {"id": 1015, "ent": "locator", "catw": "metal"}, {"id": 1016, "ent": "bassist", "catw": "living thing"}, {"id": 1017, "ent": "immune", "catw": "sport"}, {"id": 1018, "ent": "fink", "catw": "material"}, {"id": 1019, "ent": "cosmopolitan", "catw": "drink"}, {"id": 1020, "ent": "liza", "catw": "person"}, {"id": 1021, "ent": "stallion", "catw": "mammal"}, {"id": 1022, "ent": "vassal", "catw": "dog"}, {"id": 1023, "ent": "dragoon", "catw": "living thing"}, {"id": 1024, "ent": "traditionalist", "catw": "body part"}, {"id": 1025, "ent": "stallion", "catw": "living thing"}, {"id": 1026, "ent": "headmaster", "catw": "container"}, {"id": 1027, "ent": "tennis", "catw": "container"}, {"id": 1028, "ent": "godmother", "catw": "material"}, {"id": 1029, "ent": "magdalen", "catw": "boat"}, {"id": 1030, "ent": "pervert", "catw": "device"}, {"id": 1031, "ent": "neoliberal", "catw": "tool"}, {"id": 1032, "ent": "villager", "catw": "living thing"}, {"id": 1033, "ent": "salad", "catw": "machine"}, {"id": 1034, "ent": "supremacist", "catw": "person"}, {"id": 1035, "ent": "thermostat", "catw": "metal"}, {"id": 1036, "ent": "guan", "catw": "toy"}, {"id": 1037, "ent": "bodice", "catw": "clothing"}, {"id": 1038, "ent": "clarinet", "catw": "living thing"}, {"id": 1039, "ent": "tapeworm", "catw": "vegetable"}, {"id": 1040, "ent": "scold", "catw": "person"}, {"id": 1041, "ent": "ratatouille", "catw": "insect"}, {"id": 1042, "ent": "panelist", "catw": "metal"}, {"id": 1043, "ent": "vegan", "catw": "machine"}, {"id": 1044, "ent": "fern", "catw": "metal"}, {"id": 1045, "ent": "heiress", "catw": "boat"}, {"id": 1046, "ent": "mosque", "catw": "container"}, {"id": 1047, "ent": "literate", "catw": "vegetable"}, {"id": 1048, "ent": "machinist", "catw": "person"}, {"id": 1049, "ent": "sickle", "catw": "disease"}, {"id": 1050, "ent": "heiress", "catw": "fruit"}, {"id": 1051, "ent": "religious", "catw": "material"}, {"id": 1052, "ent": "spelt", "catw": "food"}, {"id": 1053, "ent": "paella", "catw": "body part"}, {"id": 1054, "ent": "laser", "catw": "game"}, {"id": 1055, "ent": "tamer", "catw": "dish"}, {"id": 1056, "ent": "executor", "catw": "worker"}, {"id": 1057, "ent": "islander", "catw": "mammal"}, {"id": 1058, "ent": "urchin", "catw": "worker"}, {"id": 1059, "ent": "obsessive", "catw": "car"}, {"id": 1060, "ent": "civilian", "catw": "tree"}, {"id": 1061, "ent": "sword", "catw": "man-made object"}, {"id": 1062, "ent": "slingshot", "catw": "musical instrument"}, {"id": 1063, "ent": "cappuccino", "catw": "living thing"}, {"id": 1064, "ent": "investor", "catw": "mammal"}, {"id": 1065, "ent": "betrothed", "catw": "game"}, {"id": 1066, "ent": "condor", "catw": "animal"}, {"id": 1067, "ent": "magistrate", "catw": "living thing"}, {"id": 1068, "ent": "turtleneck", "catw": "plant"}, {"id": 1069, "ent": "mastodon", "catw": "tool"}, {"id": 1070, "ent": "luge", "catw": "sport"}, {"id": 1071, "ent": "urologist", "catw": "person"}, {"id": 1072, "ent": "tray", "catw": "dish"}, {"id": 1073, "ent": "landlord", "catw": "game"}, {"id": 1074, "ent": "whisperer", "catw": "body part"}, {"id": 1075, "ent": "paella", "catw": "dish"}, {"id": 1076, "ent": "sandal", "catw": "room"}, {"id": 1077, "ent": "kicker", "catw": "furniture"}, {"id": 1078, "ent": "metronome", "catw": "device"}, {"id": 1079, "ent": "portable", "catw": "man-made object"}, {"id": 1080, "ent": "badminton", "catw": "container"}, {"id": 1081, "ent": "pauper", "catw": "person"}, {"id": 1082, "ent": "bigot", "catw": "game"}, {"id": 1083, "ent": "taxpayer", "catw": "vegetable"}, {"id": 1084, "ent": "anthropologist", "catw": "man-made object"}, {"id": 1085, "ent": "farmhouse", "catw": "fruit"}, {"id": 1086, "ent": "stockman", "catw": "tree"}, {"id": 1087, "ent": "tambourine", "catw": "man-made object"}, {"id": 1088, "ent": "outcast", "catw": "person"}, {"id": 1089, "ent": "contemporary", "catw": "game"}, {"id": 1090, "ent": "caliper", "catw": "reptile"}, {"id": 1091, "ent": "sprinkler", "catw": "tool"}, {"id": 1092, "ent": "screenwriter", "catw": "person"}, {"id": 1093, "ent": "lily", "catw": "reptile"}, {"id": 1094, "ent": "kinsman", "catw": "game"}, {"id": 1095, "ent": "elitist", "catw": "game"}, {"id": 1096, "ent": "archivist", "catw": "person"}, {"id": 1097, "ent": "tadpole", "catw": "building"}, {"id": 1098, "ent": "magneto", "catw": "sport"}, {"id": 1099, "ent": "catamaran", "catw": "vehicle"}, {"id": 1100, "ent": "carburetor", "catw": "animal"}, {"id": 1101, "ent": "illiterate", "catw": "living thing"}, {"id": 1102, "ent": "transformer", "catw": "sport"}, {"id": 1103, "ent": "alarmist", "catw": "person"}, {"id": 1104, "ent": "swatch", "catw": "living thing"}, {"id": 1105, "ent": "convalescent", "catw": "flower"}, {"id": 1106, "ent": "guildhall", "catw": "room"}, {"id": 1107, "ent": "sprinter", "catw": "tool"}, {"id": 1108, "ent": "pug", "catw": "man-made object"}, {"id": 1109, "ent": "practitioner", "catw": "person"}, {"id": 1110, "ent": "sheriff", "catw": "worker"}, {"id": 1111, "ent": "skier", "catw": "sport"}, {"id": 1112, "ent": "resistor", "catw": "device"}, {"id": 1113, "ent": "netball", "catw": "sport"}, {"id": 1114, "ent": "emeritus", "catw": "person"}, {"id": 1115, "ent": "spitz", "catw": "boat"}, {"id": 1116, "ent": "nettle", "catw": "equipment"}, {"id": 1117, "ent": "artiste", "catw": "body part"}, {"id": 1118, "ent": "laptop", "catw": "equipment"}, {"id": 1119, "ent": "cymbal", "catw": "animal"}, {"id": 1120, "ent": "cricketer", "catw": "person"}, {"id": 1121, "ent": "hotel", "catw": "room"}, {"id": 1122, "ent": "compatriot", "catw": "boat"}, {"id": 1123, "ent": "thistle", "catw": "flower"}, {"id": 1124, "ent": "colleen", "catw": "clothing"}, {"id": 1125, "ent": "risotto", "catw": "dish"}, {"id": 1126, "ent": "pimp", "catw": "person"}, {"id": 1127, "ent": "mantis", "catw": "animal"}, {"id": 1128, "ent": "giraffe", "catw": "fish"}, {"id": 1129, "ent": "armature", "catw": "man-made object"}, {"id": 1130, "ent": "mollie", "catw": "dog"}, {"id": 1131, "ent": "azalea", "catw": "fish"}, {"id": 1132, "ent": "condom", "catw": "device"}, {"id": 1133, "ent": "mastiff", "catw": "game"}, {"id": 1134, "ent": "butch", "catw": "artist"}, {"id": 1135, "ent": "prefect", "catw": "living thing"}, {"id": 1136, "ent": "breadwinner", "catw": "vegetable"}, {"id": 1137, "ent": "gnu", "catw": "dog"}, {"id": 1138, "ent": "supercharger", "catw": "man-made object"}, {"id": 1139, "ent": "blaster", "catw": "living thing"}, {"id": 1140, "ent": "serviceman", "catw": "material"}, {"id": 1141, "ent": "depressive", "catw": "disease"}, {"id": 1142, "ent": "collie", "catw": "animal"}, {"id": 1143, "ent": "hosiery", "catw": "man-made object"}, {"id": 1144, "ent": "buggy", "catw": "container"}, {"id": 1145, "ent": "tetra", "catw": "mammal"}, {"id": 1146, "ent": "pullover", "catw": "game"}, {"id": 1147, "ent": "yucca", "catw": "weapon"}, {"id": 1148, "ent": "asthmatic", "catw": "disease"}, {"id": 1149, "ent": "stockman", "catw": "fabric"}, {"id": 1150, "ent": "scientist", "catw": "person"}, {"id": 1151, "ent": "acacia", "catw": "tree"}, {"id": 1152, "ent": "godmother", "catw": "boat"}, {"id": 1153, "ent": "cafeteria", "catw": "building"}, {"id": 1154, "ent": "eclectic", "catw": "furniture"}, {"id": 1155, "ent": "hibiscus", "catw": "flower"}, {"id": 1156, "ent": "assemblyman", "catw": "living thing"}, {"id": 1157, "ent": "everyman", "catw": "tree"}, {"id": 1158, "ent": "sniffer", "catw": "vegetable"}, {"id": 1159, "ent": "fiance", "catw": "game"}, {"id": 1160, "ent": "caddy", "catw": "container"}, {"id": 1161, "ent": "swami", "catw": "vehicle"}, {"id": 1162, "ent": "reptile", "catw": "animal"}, {"id": 1163, "ent": "hourglass", "catw": "food"}, {"id": 1164, "ent": "baroness", "catw": "sport"}, {"id": 1165, "ent": "boyfriend", "catw": "musical instrument"}, {"id": 1166, "ent": "omelet", "catw": "weapon"}, {"id": 1167, "ent": "ant", "catw": "reptile"}, {"id": 1168, "ent": "breeder", "catw": "room"}, {"id": 1169, "ent": "lidar", "catw": "footwear"}, {"id": 1170, "ent": "penguin", "catw": "plant"}, {"id": 1171, "ent": "batik", "catw": "metal"}, {"id": 1172, "ent": "pauper", "catw": "living thing"}, {"id": 1173, "ent": "mahatma", "catw": "bird"}, {"id": 1174, "ent": "diocesan", "catw": "person"}, {"id": 1175, "ent": "shrub", "catw": "game"}, {"id": 1176, "ent": "padlock", "catw": "tool"}, {"id": 1177, "ent": "astrologer", "catw": "animal"}, {"id": 1178, "ent": "harpsichord", "catw": "equipment"}, {"id": 1179, "ent": "mediator", "catw": "living thing"}, {"id": 1180, "ent": "fanatic", "catw": "material"}, {"id": 1181, "ent": "tamer", "catw": "sport"}, {"id": 1182, "ent": "weevil", "catw": "person"}, {"id": 1183, "ent": "panelist", "catw": "living thing"}, {"id": 1184, "ent": "hacksaw", "catw": "vegetable"}, {"id": 1185, "ent": "applicant", "catw": "living thing"}, {"id": 1186, "ent": "reactionary", "catw": "boat"}, {"id": 1187, "ent": "tern", "catw": "animal"}, {"id": 1188, "ent": "registrant", "catw": "man-made object"}, {"id": 1189, "ent": "marigold", "catw": "person"}, {"id": 1190, "ent": "barometer", "catw": "device"}, {"id": 1191, "ent": "paratrooper", "catw": "insect"}, {"id": 1192, "ent": "chump", "catw": "game"}, {"id": 1193, "ent": "vertebrate", "catw": "mammal"}, {"id": 1194, "ent": "slingshot", "catw": "fish"}, {"id": 1195, "ent": "walrus", "catw": "fruit"}, {"id": 1196, "ent": "workaholic", "catw": "worker"}, {"id": 1197, "ent": "guesthouse", "catw": "metal"}, {"id": 1198, "ent": "housemate", "catw": "musical instrument"}, {"id": 1199, "ent": "groupie", "catw": "person"}, {"id": 1200, "ent": "abalone", "catw": "food"}, {"id": 1201, "ent": "seller", "catw": "artist"}, {"id": 1202, "ent": "grenade", "catw": "man-made object"}, {"id": 1203, "ent": "zebra", "catw": "vegetable"}, {"id": 1204, "ent": "serviceman", "catw": "tree"}, {"id": 1205, "ent": "colonel", "catw": "living thing"}, {"id": 1206, "ent": "colonial", "catw": "metal"}, {"id": 1207, "ent": "brew", "catw": "disease"}, {"id": 1208, "ent": "guan", "catw": "bird"}, {"id": 1209, "ent": "skiff", "catw": "container"}, {"id": 1210, "ent": "gander", "catw": "living thing"}, {"id": 1211, "ent": "cockatoo", "catw": "reptile"}, {"id": 1212, "ent": "shaman", "catw": "living thing"}, {"id": 1213, "ent": "businesswoman", "catw": "worker"}, {"id": 1214, "ent": "nomad", "catw": "worker"}, {"id": 1215, "ent": "armature", "catw": "equipment"}, {"id": 1216, "ent": "turbine", "catw": "man-made object"}, {"id": 1217, "ent": "scythe", "catw": "equipment"}, {"id": 1218, "ent": "ale", "catw": "drink"}, {"id": 1219, "ent": "cappuccino", "catw": "person"}, {"id": 1220, "ent": "vine", "catw": "disease"}, {"id": 1221, "ent": "bookshelf", "catw": "furniture"}, {"id": 1222, "ent": "dirk", "catw": "worker"}, {"id": 1223, "ent": "essayist", "catw": "material"}, {"id": 1224, "ent": "clarion", "catw": "metal"}, {"id": 1225, "ent": "boudoir", "catw": "game"}, {"id": 1226, "ent": "adjuster", "catw": "flower"}, {"id": 1227, "ent": "airbrush", "catw": "building"}, {"id": 1228, "ent": "schmuck", "catw": "material"}, {"id": 1229, "ent": "mystic", "catw": "room"}, {"id": 1230, "ent": "transponder", "catw": "equipment"}, {"id": 1231, "ent": "purist", "catw": "drink"}, {"id": 1232, "ent": "rani", "catw": "mammal"}, {"id": 1233, "ent": "overseer", "catw": "living thing"}, {"id": 1234, "ent": "locator", "catw": "tool"}, {"id": 1235, "ent": "pinto", "catw": "flower"}, {"id": 1236, "ent": "reeve", "catw": "bird"}, {"id": 1237, "ent": "valet", "catw": "device"}, {"id": 1238, "ent": "evergreen", "catw": "plant"}, {"id": 1239, "ent": "fisherman", "catw": "game"}, {"id": 1240, "ent": "chipmunk", "catw": "mammal"}, {"id": 1241, "ent": "magdalen", "catw": "person"}, {"id": 1242, "ent": "sahib", "catw": "vegetable"}, {"id": 1243, "ent": "marauder", "catw": "boat"}, {"id": 1244, "ent": "duce", "catw": "living thing"}, {"id": 1245, "ent": "bobby", "catw": "animal"}, {"id": 1246, "ent": "scapegoat", "catw": "bird"}, {"id": 1247, "ent": "sinner", "catw": "game"}, {"id": 1248, "ent": "mastodon", "catw": "flower"}, {"id": 1249, "ent": "breeder", "catw": "person"}, {"id": 1250, "ent": "bathtub", "catw": "equipment"}, {"id": 1251, "ent": "infielder", "catw": "food"}, {"id": 1252, "ent": "mustang", "catw": "living thing"}, {"id": 1253, "ent": "surfer", "catw": "athlete"}, {"id": 1254, "ent": "converter", "catw": "equipment"}, {"id": 1255, "ent": "prior", "catw": "living thing"}, {"id": 1256, "ent": "prowler", "catw": "animal"}, {"id": 1257, "ent": "juror", "catw": "worker"}, {"id": 1258, "ent": "dermatologist", "catw": "material"}, {"id": 1259, "ent": "policeman", "catw": "food"}, {"id": 1260, "ent": "triceps", "catw": "weapon"}, {"id": 1261, "ent": "tun", "catw": "boat"}, {"id": 1262, "ent": "speedometer", "catw": "equipment"}, {"id": 1263, "ent": "beggar", "catw": "sport"}, {"id": 1264, "ent": "workaholic", "catw": "bird"}, {"id": 1265, "ent": "lathe", "catw": "device"}, {"id": 1266, "ent": "gland", "catw": "organ"}, {"id": 1267, "ent": "starling", "catw": "vegetable"}, {"id": 1268, "ent": "ladle", "catw": "tool"}, {"id": 1269, "ent": "pontifical", "catw": "insect"}, {"id": 1270, "ent": "collie", "catw": "vegetable"}, {"id": 1271, "ent": "dramatist", "catw": "equipment"}, {"id": 1272, "ent": "goldfish", "catw": "animal"}, {"id": 1273, "ent": "corporal", "catw": "weapon"}, {"id": 1274, "ent": "overseer", "catw": "worker"}, {"id": 1275, "ent": "shill", "catw": "artist"}, {"id": 1276, "ent": "prisoner", "catw": "game"}, {"id": 1277, "ent": "contemporary", "catw": "artist"}, {"id": 1278, "ent": "gorilla", "catw": "athlete"}, {"id": 1279, "ent": "cursor", "catw": "tool"}, {"id": 1280, "ent": "socialite", "catw": "living thing"}, {"id": 1281, "ent": "milo", "catw": "living thing"}, {"id": 1282, "ent": "techie", "catw": "person"}, {"id": 1283, "ent": "madder", "catw": "living thing"}, {"id": 1284, "ent": "troublemaker", "catw": "vegetable"}, {"id": 1285, "ent": "bricklayer", "catw": "athlete"}, {"id": 1286, "ent": "guitar", "catw": "musical instrument"}, {"id": 1287, "ent": "fizz", "catw": "game"}, {"id": 1288, "ent": "pantry", "catw": "room"}, {"id": 1289, "ent": "bisexual", "catw": "living thing"}, {"id": 1290, "ent": "jaguar", "catw": "animal"}, {"id": 1291, "ent": "layman", "catw": "plant"}, {"id": 1292, "ent": "namesake", "catw": "sport"}, {"id": 1293, "ent": "moped", "catw": "container"}, {"id": 1294, "ent": "megaphone", "catw": "food"}, {"id": 1295, "ent": "skeptic", "catw": "boat"}, {"id": 1296, "ent": "novelist", "catw": "artist"}, {"id": 1297, "ent": "condom", "catw": "equipment"}, {"id": 1298, "ent": "yacht", "catw": "dog"}, {"id": 1299, "ent": "vine", "catw": "plant"}]''')
KIT = json.loads(r'''[{"id": 6847, "ent": "equatorial", "catw": "fabric"}, {"id": 2376, "ent": "brandy", "catw": "toy"}, {"id": 13400, "ent": "overlord", "catw": "dish"}, {"id": 11719, "ent": "matador", "catw": "worker"}, {"id": 2514, "ent": "briefcase", "catw": "toy"}, {"id": 22326, "ent": "welder", "catw": "man-made object"}, {"id": 7717, "ent": "gasket", "catw": "worker"}, {"id": 18858, "ent": "steamboat", "catw": "game"}, {"id": 22234, "ent": "waveguide", "catw": "man-made object"}, {"id": 548, "ent": "anarchist", "catw": "organ"}, {"id": 11501, "ent": "mariposa", "catw": "animal"}, {"id": 680, "ent": "aquarium", "catw": "artist"}, {"id": 20166, "ent": "telegraph", "catw": "device"}, {"id": 17297, "ent": "senna", "catw": "animal"}, {"id": 6854, "ent": "erica", "catw": "living thing"}, {"id": 11546, "ent": "married", "catw": "living thing"}, {"id": 20304, "ent": "thermos", "catw": "device"}, {"id": 20996, "ent": "turbine", "catw": "equipment"}, {"id": 11697, "ent": "matador", "catw": "food"}, {"id": 3346, "ent": "carbine", "catw": "person"}, {"id": 19782, "ent": "syringe", "catw": "mammal"}, {"id": 11470, "ent": "margarita", "catw": "man-made object"}, {"id": 17850, "ent": "skiff", "catw": "mammal"}, {"id": 6281, "ent": "dragoon", "catw": "weapon"}, {"id": 6231, "ent": "dowager", "catw": "vehicle"}, {"id": 6507, "ent": "dug", "catw": "vehicle"}, {"id": 14491, "ent": "piglet", "catw": "animal"}, {"id": 3649, "ent": "chateau", "catw": "drink"}, {"id": 4391, "ent": "coachman", "catw": "vehicle"}, {"id": 15730, "ent": "quitter", "catw": "game"}, {"id": 10646, "ent": "liberator", "catw": "equipment"}, {"id": 4712, "ent": "condor", "catw": "equipment"}, {"id": 9040, "ent": "helicopter", "catw": "aircraft"}, {"id": 18492, "ent": "sprinter", "catw": "living thing"}, {"id": 11155, "ent": "mahatma", "catw": "boat"}, {"id": 14943, "ent": "poi", "catw": "fabric"}, {"id": 13630, "ent": "palmyra", "catw": "dish"}, {"id": 18117, "ent": "socialist", "catw": "fabric"}, {"id": 1630, "ent": "belting", "catw": "equipment"}, {"id": 5425, "ent": "curry", "catw": "disease"}, {"id": 8876, "ent": "harmonica", "catw": "game"}, {"id": 21184, "ent": "undergraduate", "catw": "aircraft"}, {"id": 4601, "ent": "composer", "catw": "animal"}, {"id": 13888, "ent": "pasha", "catw": "organ"}, {"id": 10054, "ent": "interceptor", "catw": "container"}, {"id": 20956, "ent": "tun", "catw": "container"}, {"id": 9271, "ent": "houseboat", "catw": "weapon"}, {"id": 8710, "ent": "handcuff", "catw": "man-made object"}, {"id": 4145, "ent": "clary", "catw": "insect"}, {"id": 21827, "ent": "voyeur", "catw": "boat"}, {"id": 21241, "ent": "undersecretary", "catw": "worker"}, {"id": 15895, "ent": "raglan", "catw": "weapon"}, {"id": 11053, "ent": "macho", "catw": "food"}, {"id": 8774, "ent": "handyman", "catw": "person"}, {"id": 11844, "ent": "mensch", "catw": "car"}, {"id": 7933, "ent": "georgette", "catw": "vehicle"}, {"id": 15080, "ent": "portable", "catw": "material"}, {"id": 16941, "ent": "scapegoat", "catw": "food"}, {"id": 12716, "ent": "motorcycle", "catw": "equipment"}, {"id": 4515, "ent": "collie", "catw": "dog"}, {"id": 16753, "ent": "sarcophagus", "catw": "tree"}, {"id": 2300, "ent": "bramble", "catw": "living thing"}, {"id": 4910, "ent": "convalescent", "catw": "person"}, {"id": 22588, "ent": "wombat", "catw": "mammal"}, {"id": 19932, "ent": "takin", "catw": "dish"}, {"id": 12878, "ent": "narc", "catw": "game"}, {"id": 4800, "ent": "conger", "catw": "man-made object"}, {"id": 21357, "ent": "untouchable", "catw": "food"}, {"id": 7656, "ent": "fuchsia", "catw": "equipment"}, {"id": 17988, "ent": "snoop", "catw": "mammal"}, {"id": 6095, "ent": "dinky", "catw": "boat"}, {"id": 4002, "ent": "churn", "catw": "living thing"}, {"id": 10039, "ent": "interceptor", "catw": "fruit"}, {"id": 3880, "ent": "choreographer", "catw": "man-made object"}, {"id": 11239, "ent": "mamba", "catw": "drink"}, {"id": 14024, "ent": "pelican", "catw": "metal"}, {"id": 15625, "ent": "pulley", "catw": "musical instrument"}, {"id": 17546, "ent": "shim", "catw": "equipment"}, {"id": 2162, "ent": "bot", "catw": "living thing"}, {"id": 7868, "ent": "genitalia", "catw": "mammal"}, {"id": 14260, "ent": "philanthropist", "catw": "living thing"}, {"id": 13264, "ent": "notary", "catw": "man-made object"}, {"id": 20342, "ent": "thwart", "catw": "flower"}, {"id": 811, "ent": "aristocrat", "catw": "footwear"}, {"id": 10333, "ent": "jester", "catw": "footwear"}, {"id": 12098, "ent": "minion", "catw": "living thing"}, {"id": 17197, "ent": "sectional", "catw": "fabric"}, {"id": 17082, "ent": "scythe", "catw": "man-made object"}, {"id": 16347, "ent": "rifle", "catw": "tool"}, {"id": 257, "ent": "adjuster", "catw": "furniture"}, {"id": 17836, "ent": "sieve", "catw": "person"}, {"id": 10377, "ent": "juggler", "catw": "furniture"}, {"id": 1712, "ent": "betrothed", "catw": "flower"}, {"id": 21190, "ent": "undergraduate", "catw": "toy"}, {"id": 14418, "ent": "piccolo", "catw": "equipment"}, {"id": 19394, "ent": "suiting", "catw": "clothing"}, {"id": 12833, "ent": "mustang", "catw": "sport"}, {"id": 18477, "ent": "sprinkler", "catw": "musical instrument"}, {"id": 10296, "ent": "jasmine", "catw": "material"}, {"id": 9225, "ent": "hooch", "catw": "weapon"}, {"id": 19083, "ent": "stingray", "catw": "fabric"}, {"id": 22139, "ent": "washroom", "catw": "food"}, {"id": 20472, "ent": "toddler", "catw": "mammal"}, {"id": 886, "ent": "atheist", "catw": "vegetable"}, {"id": 19161, "ent": "stoner", "catw": "weapon"}, {"id": 4410, "ent": "coachman", "catw": "metal"}, {"id": 16732, "ent": "sandwich", "catw": "person"}, {"id": 9311, "ent": "housekeeper", "catw": "machine"}, {"id": 5114, "ent": "countess", "catw": "plant"}, {"id": 21770, "ent": "visitor", "catw": "vegetable"}, {"id": 13110, "ent": "neutral", "catw": "living thing"}, {"id": 19535, "ent": "supremacist", "catw": "musical instrument"}, {"id": 16652, "ent": "salesperson", "catw": "living thing"}, {"id": 13598, "ent": "paisley", "catw": "clothing"}, {"id": 21668, "ent": "veronica", "catw": "mammal"}, {"id": 6916, "ent": "espresso", "catw": "man-made object"}, {"id": 2605, "ent": "brogan", "catw": "footwear"}, {"id": 13930, "ent": "patriot", "catw": "material"}, {"id": 15932, "ent": "rapier", "catw": "man-made object"}, {"id": 3252, "ent": "capacitor", "catw": "building"}, {"id": 20872, "ent": "trident", "catw": "person"}, {"id": 12670, "ent": "motorboat", "catw": "equipment"}, {"id": 20717, "ent": "transponder", "catw": "tool"}, {"id": 16799, "ent": "sarin", "catw": "tree"}, {"id": 12511, "ent": "moll", "catw": "sport"}, {"id": 21425, "ent": "urchin", "catw": "worker"}, {"id": 9945, "ent": "insider", "catw": "tree"}, {"id": 11614, "ent": "masjid", "catw": "car"}, {"id": 859, "ent": "astrophysicist", "catw": "musical instrument"}, {"id": 13973, "ent": "pedophile", "catw": "worker"}, {"id": 2656, "ent": "buggy", "catw": "person"}, {"id": 18236, "ent": "spandex", "catw": "equipment"}, {"id": 1767, "ent": "bicycle", "catw": "machine"}, {"id": 12353, "ent": "misfit", "catw": "weapon"}, {"id": 4083, "ent": "claimant", "catw": "worker"}, {"id": 20772, "ent": "treasurer", "catw": "container"}, {"id": 172, "ent": "accountant", "catw": "person"}, {"id": 11410, "ent": "marc", "catw": "mammal"}, {"id": 21728, "ent": "vise", "catw": "man-made object"}, {"id": 12505, "ent": "moll", "catw": "fabric"}, {"id": 1855, "ent": "bimbo", "catw": "drink"}, {"id": 18671, "ent": "starfish", "catw": "body part"}, {"id": 9826, "ent": "incurable", "catw": "clothing"}, {"id": 11225, "ent": "mamba", "catw": "animal"}, {"id": 18966, "ent": "stepfather", "catw": "dish"}, {"id": 4940, "ent": "convert", "catw": "device"}, {"id": 8897, "ent": "harrow", "catw": "machine"}, {"id": 12577, "ent": "moped", "catw": "machine"}, {"id": 2476, "ent": "breeder", "catw": "material"}, {"id": 7167, "ent": "extrovert", "catw": "athlete"}, {"id": 20693, "ent": "towel", "catw": "fabric"}, {"id": 4760, "ent": "confidante", "catw": "car"}, {"id": 12190, "ent": "minnow", "catw": "living thing"}, {"id": 15858, "ent": "radiologist", "catw": "person"}, {"id": 22623, "ent": "wombat", "catw": "athlete"}, {"id": 18378, "ent": "spitfire", "catw": "aircraft"}, {"id": 2529, "ent": "briefcase", "catw": "sport"}, {"id": 11932, "ent": "messenger", "catw": "device"}, {"id": 12383, "ent": "missus", "catw": "tree"}, {"id": 1956, "ent": "bistro", "catw": "aircraft"}, {"id": 17768, "ent": "sickle", "catw": "vegetable"}, {"id": 705, "ent": "archbishop", "catw": "drink"}, {"id": 8612, "ent": "grocer", "catw": "flower"}, {"id": 12106, "ent": "minion", "catw": "plant"}, {"id": 10727, "ent": "lilac", "catw": "tree"}, {"id": 1138, "ent": "baboon", "catw": "person"}, {"id": 2157, "ent": "bodybuilder", "catw": "body part"}, {"id": 13033, "ent": "necktie", "catw": "drink"}, {"id": 10992, "ent": "lodger", "catw": "game"}, {"id": 1488, "ent": "beanie", "catw": "man-made object"}, {"id": 8539, "ent": "greenhouse", "catw": "footwear"}, {"id": 19560, "ent": "surrealist", "catw": "flower"}, {"id": 7281, "ent": "fancier", "catw": "food"}, {"id": 12009, "ent": "migrant", "catw": "bird"}, {"id": 17019, "ent": "scion", "catw": "sport"}, {"id": 4109, "ent": "clarence", "catw": "drink"}, {"id": 17265, "ent": "senator", "catw": "drink"}, {"id": 7072, "ent": "exhibitor", "catw": "person"}, {"id": 2852, "ent": "businesswoman", "catw": "living thing"}, {"id": 7107, "ent": "exporter", "catw": "boat"}, {"id": 3190, "ent": "candlestick", "catw": "man-made object"}, {"id": 8360, "ent": "granddaughter", "catw": "person"}, {"id": 820, "ent": "aristocrat", "catw": "material"}, {"id": 7799, "ent": "gavel", "catw": "weapon"}, {"id": 15022, "ent": "pontifical", "catw": "container"}, {"id": 553, "ent": "ancestor", "catw": "animal"}, {"id": 85, "ent": "absentee", "catw": "fabric"}, {"id": 10074, "ent": "invalid", "catw": "living thing"}, {"id": 13098, "ent": "neurologist", "catw": "person"}, {"id": 15111, "ent": "potion", "catw": "boat"}, {"id": 11868, "ent": "mesquite", "catw": "living thing"}, {"id": 9233, "ent": "hooch", "catw": "room"}, {"id": 406, "ent": "alumnus", "catw": "material"}, {"id": 15224, "ent": "princess", "catw": "game"}, {"id": 11380, "ent": "mangrove", "catw": "device"}, {"id": 20214, "ent": "telescope", "catw": "equipment"}, {"id": 20532, "ent": "tongs", "catw": "man-made object"}, {"id": 8489, "ent": "grandson", "catw": "weapon"}, {"id": 14392, "ent": "physiologist", "catw": "metal"}, {"id": 18603, "ent": "stapler", "catw": "machine"}, {"id": 9642, "ent": "immigrant", "catw": "clothing"}, {"id": 6553, "ent": "dynamo", "catw": "vehicle"}, {"id": 8030, "ent": "gland", "catw": "container"}, {"id": 3970, "ent": "chub", "catw": "dish"}, {"id": 21042, "ent": "tweeter", "catw": "equipment"}, {"id": 9185, "ent": "homophobe", "catw": "musical instrument"}, {"id": 3954, "ent": "chronicler", "catw": "game"}, {"id": 13680, "ent": "panzer", "catw": "device"}, {"id": 1794, "ent": "biennial", "catw": "living thing"}, {"id": 8253, "ent": "grabber", "catw": "machine"}, {"id": 8084, "ent": "glider", "catw": "person"}, {"id": 14846, "ent": "plumber", "catw": "person"}, {"id": 18836, "ent": "steamboat", "catw": "car"}, {"id": 22412, "ent": "whiskey", "catw": "flower"}, {"id": 6392, "ent": "dredge", "catw": "game"}, {"id": 20274, "ent": "therapist", "catw": "person"}, {"id": 9522, "ent": "hyena", "catw": "living thing"}, {"id": 11640, "ent": "mastiff", "catw": "mammal"}, {"id": 1514, "ent": "beanie", "catw": "organ"}, {"id": 18401, "ent": "spokeswoman", "catw": "animal"}, {"id": 7357, "ent": "fascist", "catw": "disease"}, {"id": 11342, "ent": "manger", "catw": "container"}, {"id": 5914, "ent": "denture", "catw": "container"}, {"id": 637, "ent": "anesthesiologist", "catw": "fabric"}, {"id": 17728, "ent": "shogun", "catw": "device"}, {"id": 21025, "ent": "tweeter", "catw": "bird"}, {"id": 8574, "ent": "greyhound", "catw": "device"}, {"id": 15138, "ent": "preteen", "catw": "fish"}, {"id": 9861, "ent": "informer", "catw": "tool"}, {"id": 3574, "ent": "cesspool", "catw": "building"}, {"id": 13843, "ent": "paranoid", "catw": "disease"}, {"id": 20089, "ent": "tavern", "catw": "room"}, {"id": 17566, "ent": "shim", "catw": "metal"}, {"id": 13484, "ent": "pagoda", "catw": "reptile"}, {"id": 6325, "ent": "dreadnought", "catw": "boat"}, {"id": 26, "ent": "abolitionist", "catw": "container"}, {"id": 4110, "ent": "clarence", "catw": "man-made object"}, {"id": 2826, "ent": "burrito", "catw": "equipment"}, {"id": 14341, "ent": "physicist", "catw": "worker"}, {"id": 3405, "ent": "caribou", "catw": "animal"}, {"id": 3680, "ent": "cheetah", "catw": "living thing"}, {"id": 3053, "ent": "calender", "catw": "tool"}, {"id": 22929, "ent": "yacht", "catw": "vehicle"}, {"id": 3068, "ent": "calender", "catw": "building"}, {"id": 20450, "ent": "toby", "catw": "container"}, {"id": 19425, "ent": "sunflower", "catw": "food"}, {"id": 21572, "ent": "valet", "catw": "game"}, {"id": 19716, "ent": "swine", "catw": "clothing"}, {"id": 19751, "ent": "swot", "catw": "tool"}, {"id": 17041, "ent": "scraper", "catw": "vehicle"}, {"id": 18205, "ent": "sofa", "catw": "worker"}, {"id": 17191, "ent": "sectional", "catw": "room"}, {"id": 8126, "ent": "godmother", "catw": "toy"}, {"id": 1844, "ent": "bimbo", "catw": "fish"}, {"id": 16394, "ent": "rival", "catw": "device"}, {"id": 5921, "ent": "denture", "catw": "room"}, {"id": 16482, "ent": "rogue", "catw": "dish"}, {"id": 1669, "ent": "betel", "catw": "food"}, {"id": 10921, "ent": "locksmith", "catw": "machine"}, {"id": 7544, "ent": "foal", "catw": "living thing"}, {"id": 402, "ent": "alumnus", "catw": "person"}, {"id": 22657, "ent": "woodpecker", "catw": "weapon"}, {"id": 6432, "ent": "dropper", "catw": "material"}, {"id": 17592, "ent": "ship", "catw": "equipment"}, {"id": 7831, "ent": "geisha", "catw": "fruit"}, {"id": 9482, "ent": "husband", "catw": "reptile"}, {"id": 2440, "ent": "breeder", "catw": "mammal"}, {"id": 12941, "ent": "national", "catw": "drink"}, {"id": 11757, "ent": "maul", "catw": "furniture"}, {"id": 9401, "ent": "huck", "catw": "tool"}, {"id": 2776, "ent": "bumblebee", "catw": "man-made object"}, {"id": 14238, "ent": "peripheral", "catw": "aircraft"}, {"id": 6825, "ent": "equatorial", "catw": "tool"}, {"id": 9462, "ent": "hummingbird", "catw": "building"}, {"id": 9127, "ent": "hoarder", "catw": "machine"}, {"id": 7409, "ent": "finch", "catw": "bird"}, {"id": 22824, "ent": "wrestler", "catw": "plant"}, {"id": 11010, "ent": "loom", "catw": "man-made object"}, {"id": 8878, "ent": "harrow", "catw": "living thing"}, {"id": 11111, "ent": "maggot", "catw": "weapon"}, {"id": 13010, "ent": "nawab", "catw": "material"}, {"id": 22534, "ent": "wildflower", "catw": "metal"}, {"id": 1460, "ent": "beagle", "catw": "person"}, {"id": 2908, "ent": "buttercup", "catw": "flower"}, {"id": 1929, "ent": "biographer", "catw": "disease"}, {"id": 14658, "ent": "playmate", "catw": "toy"}, {"id": 221, "ent": "accused", "catw": "athlete"}, {"id": 16942, "ent": "scapegoat", "catw": "dish"}, {"id": 987, "ent": "avenger", "catw": "vehicle"}, {"id": 19300, "ent": "stray", "catw": "container"}, {"id": 16291, "ent": "ridley", "catw": "dog"}, {"id": 18148, "ent": "socialite", "catw": "aircraft"}, {"id": 22283, "ent": "weatherman", "catw": "machine"}, {"id": 7383, "ent": "fife", "catw": "boat"}, {"id": 21148, "ent": "underdog", "catw": "person"}, {"id": 16073, "ent": "recliner", "catw": "machine"}, {"id": 18558, "ent": "stagecoach", "catw": "equipment"}, {"id": 8662, "ent": "haggis", "catw": "dish"}, {"id": 4489, "ent": "coffin", "catw": "furniture"}, {"id": 10785, "ent": "limousine", "catw": "vehicle"}, {"id": 1097, "ent": "azalea", "catw": "fabric"}, {"id": 17410, "ent": "shed", "catw": "car"}, {"id": 17245, "ent": "semaphore", "catw": "body part"}, {"id": 22025, "ent": "warden", "catw": "athlete"}, {"id": 2181, "ent": "bot", "catw": "machine"}, {"id": 18908, "ent": "stepdaughter", "catw": "mammal"}, {"id": 13313, "ent": "oboe", "catw": "machine"}, {"id": 5013, "ent": "corgi", "catw": "sport"}, {"id": 14966, "ent": "poncho", "catw": "man-made object"}, {"id": 10560, "ent": "landlord", "catw": "container"}, {"id": 3647, "ent": "chateau", "catw": "food"}, {"id": 8896, "ent": "harrow", "catw": "device"}, {"id": 20115, "ent": "teapot", "catw": "food"}, {"id": 5247, "ent": "cricketer", "catw": "bird"}, {"id": 5732, "ent": "debtor", "catw": "clothing"}, {"id": 15166, "ent": "preteen", "catw": "building"}, {"id": 15502, "ent": "psychotherapist", "catw": "living thing"}, {"id": 10786, "ent": "limousine", "catw": "car"}, {"id": 5094, "ent": "councilman", "catw": "person"}, {"id": 922, "ent": "attacker", "catw": "mammal"}, {"id": 6647, "ent": "educator", "catw": "boat"}, {"id": 15978, "ent": "ratchet", "catw": "man-made object"}, {"id": 15493, "ent": "psychiatrist", "catw": "athlete"}, {"id": 12518, "ent": "mongoose", "catw": "reptile"}, {"id": 4708, "ent": "condor", "catw": "man-made object"}, {"id": 19180, "ent": "stoner", "catw": "game"}, {"id": 17638, "ent": "shirt", "catw": "equipment"}, {"id": 18337, "ent": "speakeasy", "catw": "footwear"}, {"id": 1313, "ent": "banjo", "catw": "weapon"}, {"id": 2734, "ent": "bulldozer", "catw": "equipment"}, {"id": 3979, "ent": "chub", "catw": "boat"}, {"id": 3861, "ent": "chisel", "catw": "disease"}, {"id": 17454, "ent": "sheeting", "catw": "equipment"}, {"id": 5825, "ent": "deflector", "catw": "footwear"}, {"id": 22244, "ent": "waveguide", "catw": "container"}, {"id": 6703, "ent": "egalitarian", "catw": "room"}, {"id": 7518, "ent": "flashlight", "catw": "equipment"}, {"id": 6044, "ent": "dimmer", "catw": "device"}, {"id": 11222, "ent": "mako", "catw": "game"}, {"id": 15559, "ent": "pug", "catw": "fruit"}, {"id": 15448, "ent": "prowler", "catw": "material"}, {"id": 5503, "ent": "cutthroat", "catw": "footwear"}, {"id": 17748, "ent": "shogun", "catw": "material"}, {"id": 2712, "ent": "bulldog", "catw": "game"}, {"id": 22186, "ent": "watchman", "catw": "dish"}, {"id": 17407, "ent": "shed", "catw": "machine"}, {"id": 2072, "ent": "bobcat", "catw": "mammal"}, {"id": 17911, "ent": "sled", "catw": "tool"}, {"id": 14294, "ent": "philanthropist", "catw": "person"}, {"id": 6183, "ent": "dongle", "catw": "machine"}, {"id": 16622, "ent": "sabra", "catw": "man-made object"}, {"id": 16887, "ent": "scalpel", "catw": "insect"}, {"id": 5290, "ent": "crocodile", "catw": "living thing"}, {"id": 330, "ent": "althea", "catw": "plant"}, {"id": 14887, "ent": "plush", "catw": "footwear"}, {"id": 6724, "ent": "empress", "catw": "plant"}, {"id": 4337, "ent": "clover", "catw": "food"}, {"id": 22758, "ent": "worthy", "catw": "person"}, {"id": 5996, "ent": "desk", "catw": "man-made object"}, {"id": 18000, "ent": "snoop", "catw": "dish"}, {"id": 19493, "ent": "supermodel", "catw": "worker"}, {"id": 15564, "ent": "pug", "catw": "man-made object"}, {"id": 307, "ent": "aircraft", "catw": "musical instrument"}, {"id": 5551, "ent": "cymbal", "catw": "musical instrument"}, {"id": 16168, "ent": "retailer", "catw": "car"}, {"id": 1022, "ent": "ax", "catw": "flower"}, {"id": 14897, "ent": "plush", "catw": "fabric"}, {"id": 6670, "ent": "egalitarian", "catw": "living thing"}, {"id": 6160, "ent": "dirk", "catw": "organ"}, {"id": 18701, "ent": "starling", "catw": "weapon"}, {"id": 479, "ent": "amplifier", "catw": "machine"}, {"id": 19596, "ent": "survivalist", "catw": "living thing"}, {"id": 10672, "ent": "librarian", "catw": "living thing"}, {"id": 11992, "ent": "midshipman", "catw": "building"}, {"id": 12432, "ent": "modernist", "catw": "vegetable"}, {"id": 1518, "ent": "bedroom", "catw": "living thing"}, {"id": 8195, "ent": "goldfish", "catw": "dog"}, {"id": 14178, "ent": "perfectionist", "catw": "flower"}, {"id": 16244, "ent": "rhinoceros", "catw": "reptile"}, {"id": 7241, "ent": "falcon", "catw": "machine"}, {"id": 18901, "ent": "steeper", "catw": "body part"}, {"id": 13382, "ent": "ombudsman", "catw": "organ"}, {"id": 12070, "ent": "mimic", "catw": "device"}, {"id": 10825, "ent": "liza", "catw": "drink"}, {"id": 22482, "ent": "wife", "catw": "person"}, {"id": 2530, "ent": "broad", "catw": "living thing"}, {"id": 11427, "ent": "marc", "catw": "machine"}, {"id": 22962, "ent": "yucca", "catw": "plant"}, {"id": 16228, "ent": "revisionist", "catw": "artist"}, {"id": 3309, "ent": "capo", "catw": "disease"}, {"id": 12282, "ent": "miser", "catw": "living thing"}, {"id": 12160, "ent": "minivan", "catw": "man-made object"}, {"id": 18258, "ent": "spandex", "catw": "organ"}, {"id": 8445, "ent": "grandparent", "catw": "furniture"}, {"id": 4692, "ent": "condor", "catw": "living thing"}, {"id": 16558, "ent": "rowan", "catw": "game"}, {"id": 19911, "ent": "tailor", "catw": "fabric"}, {"id": 16130, "ent": "redneck", "catw": "toy"}, {"id": 3450, "ent": "celibate", "catw": "living thing"}, {"id": 10441, "ent": "kicker", "catw": "sport"}, {"id": 111, "ent": "accelerometer", "catw": "machine"}, {"id": 21470, "ent": "valedictorian", "catw": "person"}, {"id": 9912, "ent": "injector", "catw": "car"}, {"id": 21305, "ent": "undies", "catw": "dog"}, {"id": 17955, "ent": "slob", "catw": "drink"}, {"id": 3154, "ent": "camellia", "catw": "container"}, {"id": 11747, "ent": "maul", "catw": "tool"}, {"id": 14052, "ent": "penis", "catw": "car"}, {"id": 1590, "ent": "beekeeper", "catw": "container"}, {"id": 23005, "ent": "zebra", "catw": "insect"}, {"id": 11334, "ent": "manger", "catw": "device"}, {"id": 772, "ent": "architect", "catw": "artist"}, {"id": 8269, "ent": "grabber", "catw": "worker"}, {"id": 5794, "ent": "deer", "catw": "game"}, {"id": 5628, "ent": "darkroom", "catw": "man-made object"}, {"id": 22551, "ent": "windowsill", "catw": "fruit"}, {"id": 21970, "ent": "wanton", "catw": "clothing"}, {"id": 7613, "ent": "forceps", "catw": "boat"}, {"id": 16708, "ent": "sandwich", "catw": "flower"}, {"id": 7492, "ent": "flamingo", "catw": "metal"}, {"id": 16072, "ent": "recliner", "catw": "device"}, {"id": 17682, "ent": "shiv", "catw": "device"}, {"id": 9740, "ent": "importer", "catw": "person"}, {"id": 18374, "ent": "spitfire", "catw": "equipment"}, {"id": 6344, "ent": "dreadnought", "catw": "organ"}, {"id": 10155, "ent": "investor", "catw": "worker"}, {"id": 2944, "ent": "buttermilk", "catw": "living thing"}, {"id": 13190, "ent": "newsreader", "catw": "person"}, {"id": 10488, "ent": "krill", "catw": "living thing"}, {"id": 5736, "ent": "debtor", "catw": "building"}, {"id": 16437, "ent": "rodent", "catw": "drink"}, {"id": 20818, "ent": "tricycle", "catw": "container"}, {"id": 20324, "ent": "thermos", "catw": "material"}, {"id": 1921, "ent": "biographer", "catw": "worker"}, {"id": 6227, "ent": "dowager", "catw": "tool"}, {"id": 6969, "ent": "ethnic", "catw": "boat"}, {"id": 13540, "ent": "paintbrush", "catw": "man-made object"}, {"id": 4148, "ent": "clary", "catw": "plant"}, {"id": 14582, "ent": "playmaker", "catw": "living thing"}, {"id": 13788, "ent": "paramedic", "catw": "person"}, {"id": 14930, "ent": "poi", "catw": "container"}, {"id": 14546, "ent": "pinto", "catw": "flower"}, {"id": 2069, "ent": "boatman", "catw": "sport"}, {"id": 2120, "ent": "bodybuilder", "catw": "fish"}, {"id": 9004, "ent": "heathen", "catw": "person"}, {"id": 10474, "ent": "koala", "catw": "building"}, {"id": 4687, "ent": "conch", "catw": "body part"}, {"id": 5174, "ent": "coupe", "catw": "car"}, {"id": 1201, "ent": "backpack", "catw": "insect"}, {"id": 16844, "ent": "sauerkraut", "catw": "plant"}, {"id": 13223, "ent": "newt", "catw": "vehicle"}, {"id": 5578, "ent": "dachshund", "catw": "vegetable"}, {"id": 19212, "ent": "storeroom", "catw": "toy"}, {"id": 4309, "ent": "clockwork", "catw": "musical instrument"}, {"id": 10166, "ent": "isolationist", "catw": "living thing"}, {"id": 21616, "ent": "vaporizer", "catw": "organ"}, {"id": 18307, "ent": "spastic", "catw": "sport"}, {"id": 8090, "ent": "glider", "catw": "metal"}, {"id": 17966, "ent": "slob", "catw": "container"}, {"id": 1271, "ent": "ballerina", "catw": "footwear"}, {"id": 15343, "ent": "programmer", "catw": "weapon"}, {"id": 18021, "ent": "snoop", "catw": "worker"}, {"id": 20896, "ent": "tugboat", "catw": "vegetable"}, {"id": 12881, "ent": "narrator", "catw": "animal"}, {"id": 20048, "ent": "tartan", "catw": "material"}, {"id": 20568, "ent": "tortoise", "catw": "reptile"}, {"id": 18768, "ent": "statesman", "catw": "living thing"}, {"id": 13446, "ent": "paella", "catw": "dish"}, {"id": 20004, "ent": "tambourine", "catw": "metal"}, {"id": 12616, "ent": "moray", "catw": "vegetable"}, {"id": 19231, "ent": "stork", "catw": "bird"}, {"id": 8714, "ent": "handcuff", "catw": "equipment"}, {"id": 421, "ent": "amigo", "catw": "dog"}, {"id": 22062, "ent": "warlord", "catw": "clothing"}, {"id": 8158, "ent": "goggles", "catw": "man-made object"}, {"id": 19834, "ent": "tabor", "catw": "plant"}, {"id": 22968, "ent": "yucca", "catw": "dish"}, {"id": 5099, "ent": "councilman", "catw": "fabric"}, {"id": 6766, "ent": "environmentalist", "catw": "fish"}, {"id": 18568, "ent": "stagecoach", "catw": "toy"}, {"id": 3681, "ent": "cheetah", "catw": "animal"}, {"id": 21925, "ent": "walrus", "catw": "footwear"}, {"id": 14481, "ent": "pickpocket", "catw": "athlete"}, {"id": 14707, "ent": "pleaser", "catw": "room"}, {"id": 9773, "ent": "incubator", "catw": "vehicle"}, {"id": 18076, "ent": "soccer", "catw": "game"}, {"id": 10489, "ent": "krill", "catw": "animal"}, {"id": 13007, "ent": "nawab", "catw": "worker"}, {"id": 15239, "ent": "probable", "catw": "food"}, {"id": 3007, "ent": "caddie", "catw": "tool"}, {"id": 13671, "ent": "panzer", "catw": "tree"}, {"id": 14147, "ent": "peregrine", "catw": "weapon"}, {"id": 9600, "ent": "illustrator", "catw": "building"}, {"id": 10001, "ent": "integrator", "catw": "machine"}, {"id": 15389, "ent": "protectionist", "catw": "weapon"}, {"id": 4848, "ent": "contraceptive", "catw": "device"}, {"id": 10585, "ent": "larva", "catw": "insect"}, {"id": 14103, "ent": "peony", "catw": "furniture"}, {"id": 1357, "ent": "bantam", "catw": "boat"}, {"id": 15657, "ent": "puppeteer", "catw": "tool"}, {"id": 18291, "ent": "spastic", "catw": "footwear"}, {"id": 9372, "ent": "housemate", "catw": "person"}, {"id": 7184, "ent": "falafel", "catw": "plant"}, {"id": 2408, "ent": "brazier", "catw": "man-made object"}, {"id": 10876, "ent": "llama", "catw": "equipment"}, {"id": 17508, "ent": "sherry", "catw": "clothing"}, {"id": 1981, "ent": "boathouse", "catw": "bird"}, {"id": 5692, "ent": "dauphin", "catw": "person"}, {"id": 21885, "ent": "waitress", "catw": "worker"}, {"id": 6398, "ent": "dropper", "catw": "fish"}, {"id": 4215, "ent": "classroom", "catw": "footwear"}, {"id": 7928, "ent": "georgette", "catw": "man-made object"}, {"id": 305, "ent": "aircraft", "catw": "footwear"}, {"id": 16001, "ent": "ratchet", "catw": "fabric"}, {"id": 5468, "ent": "customer", "catw": "metal"}, {"id": 8825, "ent": "hangman", "catw": "fabric"}, {"id": 12271, "ent": "minster", "catw": "worker"}, {"id": 6622, "ent": "eclectic", "catw": "game"}, {"id": 21494, "ent": "valerian", "catw": "vegetable"}, {"id": 3786, "ent": "chipmunk", "catw": "dish"}, {"id": 19661, "ent": "swami", "catw": "machine"}, {"id": 16046, "ent": "rayon", "catw": "material"}, {"id": 19133, "ent": "stockman", "catw": "disease"}, {"id": 13847, "ent": "pasha", "catw": "animal"}, {"id": 15732, "ent": "racist", "catw": "living thing"}, {"id": 5353, "ent": "crossbow", "catw": "tool"}, {"id": 17115, "ent": "secessionist", "catw": "bird"}, {"id": 11297, "ent": "mammal", "catw": "furniture"}, {"id": 12456, "ent": "modernist", "catw": "artist"}, {"id": 3527, "ent": "cello", "catw": "musical instrument"}, {"id": 13345, "ent": "ombudsman", "catw": "insect"}, {"id": 1150, "ent": "babu", "catw": "living thing"}, {"id": 22683, "ent": "workstation", "catw": "insect"}, {"id": 3727, "ent": "chimpanzee", "catw": "animal"}, {"id": 5873, "ent": "delinquent", "catw": "musical instrument"}, {"id": 7748, "ent": "gat", "catw": "equipment"}, {"id": 3358, "ent": "cardiologist", "catw": "living thing"}, {"id": 3599, "ent": "chalet", "catw": "fruit"}, {"id": 14787, "ent": "plow", "catw": "vehicle"}, {"id": 22786, "ent": "wraparound", "catw": "man-made object"}, {"id": 9137, "ent": "hoarder", "catw": "footwear"}, {"id": 4671, "ent": "conch", "catw": "weapon"}, {"id": 10811, "ent": "liza", "catw": "animal"}, {"id": 6475, "ent": "dubbing", "catw": "worker"}, {"id": 19006, "ent": "stepmother", "catw": "plant"}, {"id": 15992, "ent": "ratchet", "catw": "toy"}, {"id": 18738, "ent": "starship", "catw": "man-made object"}, {"id": 8309, "ent": "grader", "catw": "footwear"}, {"id": 7261, "ent": "falcon", "catw": "fabric"}, {"id": 12942, "ent": "national", "catw": "man-made object"}, {"id": 4421, "ent": "coauthor", "catw": "insect"}, {"id": 12972, "ent": "nawab", "catw": "living thing"}, {"id": 10412, "ent": "kicker", "catw": "man-made object"}, {"id": 14920, "ent": "poi", "catw": "man-made object"}, {"id": 5968, "ent": "dermatologist", "catw": "person"}, {"id": 2247, "ent": "boyfriend", "catw": "fabric"}, {"id": 21636, "ent": "vermouth", "catw": "man-made object"}, {"id": 9680, "ent": "immune", "catw": "equipment"}, {"id": 21893, "ent": "waitress", "catw": "disease"}, {"id": 11776, "ent": "meatball", "catw": "living thing"}, {"id": 20394, "ent": "tights", "catw": "man-made object"}, {"id": 22882, "ent": "wristwatch", "catw": "equipment"}, {"id": 14048, "ent": "penis", "catw": "device"}, {"id": 21274, "ent": "understudy", "catw": "car"}, {"id": 19392, "ent": "suiting", "catw": "container"}, {"id": 14754, "ent": "plenipotentiary", "catw": "person"}, {"id": 10431, "ent": "kicker", "catw": "worker"}, {"id": 12755, "ent": "murine", "catw": "food"}, {"id": 5240, "ent": "cowgirl", "catw": "organ"}, {"id": 20631, "ent": "totalitarian", "catw": "boat"}, {"id": 17646, "ent": "shirt", "catw": "clothing"}, {"id": 4266, "ent": "clinician", "catw": "person"}, {"id": 10215, "ent": "janitor", "catw": "bird"}, {"id": 11718, "ent": "matador", "catw": "person"}, {"id": 22105, "ent": "warrior", "catw": "weapon"}, {"id": 2295, "ent": "brad", "catw": "body part"}, {"id": 19364, "ent": "succulent", "catw": "game"}, {"id": 12087, "ent": "mimic", "catw": "worker"}, {"id": 17387, "ent": "shaheed", "catw": "sport"}, {"id": 19629, "ent": "survivalist", "catw": "room"}, {"id": 15796, "ent": "radar", "catw": "device"}, {"id": 1381, "ent": "bayonet", "catw": "animal"}, {"id": 13725, "ent": "papillon", "catw": "tool"}, {"id": 7856, "ent": "geisha", "catw": "artist"}, {"id": 20307, "ent": "thermos", "catw": "vehicle"}, {"id": 16572, "ent": "rustic", "catw": "vegetable"}, {"id": 22372, "ent": "wheelchair", "catw": "man-made object"}, {"id": 3100, "ent": "caliper", "catw": "device"}, {"id": 8958, "ent": "headmaster", "catw": "person"}, {"id": 9086, "ent": "hillbilly", "catw": "aircraft"}, {"id": 15282, "ent": "progenitor", "catw": "flower"}, {"id": 21379, "ent": "untouchable", "catw": "worker"}, {"id": 23035, "ent": "zebra", "catw": "worker"}, {"id": 21077, "ent": "undercarriage", "catw": "tree"}, {"id": 4596, "ent": "commissar", "catw": "organ"}, {"id": 7958, "ent": "girder", "catw": "living thing"}, {"id": 6999, "ent": "evergreen", "catw": "dog"}, {"id": 5035, "ent": "cougar", "catw": "vehicle"}, {"id": 8374, "ent": "grandma", "catw": "mammal"}, {"id": 10348, "ent": "jester", "catw": "game"}]''')
SETS = {'e8': OLD + NEW, 'k': KIT}
r12 = json.load(open(E112))
L = r12['L']
CH = {'n': {(w, k): v for w, k, v in r12['choice']}, 'g': {}}
ST = {'n': {(w, k): tuple(v) for w, k, *v in r12['steps']}, 'g': {}}
AR = {(w, k, C): tuple(v) for w, k, C, *v in r12['arrive']}
with gzip.open(WIKT, 'rt', encoding='utf-8') as f: WKD = json.load(f)['n']
print(len(SETS['e8']), '問（E8）', len(SETS['k']), '問（E10-3 の一種か）  語', len(L), '  辞書の見出し', len(WKD))

## 1. NLI と Gemma を読み込む（GPU、2〜4分）

In [ ]:
import torch, spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
nlp = spacy.load("en_core_web_sm", disable=["ner"])
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORDER = [lab["entailment"], lab["neutral"], lab["contradiction"]]
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
NUM = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1, MAX_DEFS_MID + 1)]
YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]

def _nli(P, H, B=64):
    out = []
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORDER].cpu().numpy()
        out += [(round(float(p[0]), 4), round(float(p[2]), 4)) for p in pr]
    return out
def nli(trip):
    # (w, k, c)：前提「"w" means: 説明文」、仮説「W is a type of c.」（E8n と同じ）
    return _nli([premise(w, L[w][k]["def"]) for w, k, c in trip], [hyp(w, c) for w, k, c in trip])
def _pick(qs, ns, B=16):
    out = []
    for i in range(0, len(qs), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        for row, n in zip(last, ns[i:i + B]):
            out.append(int(torch.softmax(row[NUM[:n]], -1).argmax()))
    return out
def yes_no(qs, B=16):
    out = []
    for i in range(0, len(qs), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            r = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        out.extend((r[:, YES] - r[:, NO]).cpu().numpy().round(3).tolist())
    return out
print(genus_prompt("kitten", "A young cat.", ["cat"]))
print(arrive_prompt("kitten", "mammal", ['- "kitten": A young cat.', '- "cat": A small domesticated carnivorous mammal.']))
print(yes_no([arrive_prompt("kitten", "mammal", ['- "kitten": A young cat.', '- "cat": A small domesticated carnivorous mammal.']),
              arrive_prompt("kitten", "mammal", [])]))

## 2. 鎖を作る（親の語：NLI の作り方 n と Gemma の作り方 g、3段まで。GPU、10〜20分）

In [ ]:
DONE = {"n": set(), "g": set()}
def grow(ents, mode, maxd=MAXD):
    # 物の語から maxd 段まで、まだ親の語を決めていない意味だけ決める。
    # n：候補のうち NLI（説明文 ⇒ W is a type of 候補）が一番の物（E8n と同じ）。E11-2 で決めた物はそのまま使う。
    # g：候補が2つ以上なら Gemma に選ばせる（genus_prompt）。段の強さは同じ NLI の値。
    key = gkey(mode)
    front = sorted(set(ents))
    for w in front: new_word(L, w, WKD, nlp)
    for d in range(maxd):
        todo = []
        for w in front:
            for k, x in enumerate(L.get(w, [])):
                if (w, k) in DONE[mode]: continue
                DONE[mode].add((w, k))
                if mode == "n" and x.get("genus") is not None and (w, k) in ST["n"]: continue   # E11-2 で決めた物
                for c in x["cands"]:
                    if c != w: new_word(L, c, WKD, nlp)
                cs = [c for c in x["cands"] if c != w and L.get(c)]
                if cs: todo.append((w, k, cs))
                else: x[key] = None
        trip = sorted({(w, k, c) for w, k, cs in todo for c in cs if (w, k, c) not in SC})
        if trip: SC.update(zip(trip, nli(trip)))
        if mode == "n":
            for w, k, cs in todo:
                g = max(cs, key=lambda c: SC[(w, k, c)][0]); L[w][k]["genus"] = g; ST["n"][(w, k)] = SC[(w, k, g)]
        else:
            many = [(w, k, cs) for w, k, cs in todo if len(cs) > 1]
            pk = _pick([genus_prompt(w, L[w][k]["def"], cs) for w, k, cs in many], [len(cs) for w, k, cs in many]) if many else []
            for (w, k, cs), i in zip(many, pk): L[w][k]["ggenus"] = cs[i]
            for w, k, cs in todo:
                if len(cs) == 1: L[w][k]["ggenus"] = cs[0]
                ST["g"][(w, k)] = SC[(w, k, L[w][k]["ggenus"])]
        # 親の語の意味（2つ以上ある時だけ Gemma、E8n の方法D と同じ聞き方）
        jobs = [(w, k) for w, k, cs in todo if len(L[L[w][k][key]]) > 1 and (w, k) not in CH[mode]]
        if jobs:
            qs = [choice_prompt(w, L[w][k]["def"], L[w][k][key], [x["def"] for x in L[L[w][k][key]]]) for w, k in jobs]
            for (w, k), v in zip(jobs, _pick(qs, [len(L[L[w][k][key]]) for w, k in jobs])): CH[mode][(w, k)] = int(v)
        nxt = set()
        for w in front:
            for k in range(len(L.get(w, []))):
                s = step_from(L, CH, ST, mode, w, k)
                if s: nxt.add(s[0][0])
        print(f"  {mode} 段{d + 1}: 決めた意味 {len(todo)}  NLI {len(trip)}  語 {len(L)}")
        front = sorted(nxt)

saved = load_gz("E15_chains")
if saved is not None:
    L = saved["L"]
    CH = {m: {(w, k): v for w, k, v in saved["CH"][m]} for m in "ng"}
    ST = {m: {(w, k): tuple(v) for w, k, *v in saved["ST"][m]} for m in "ng"}
    print("ドライブから読み込み")
else:
    SC = {}
    t = time.time()
    grow([it["ent"] for it in SETS["k"]], "n")
    grow([it["ent"] for s in SETS.values() for it in s], "g")
    save_gz({"L": L, "CH": {m: [[w, k, v] for (w, k), v in CH[m].items()] for m in "ng"},
             "ST": {m: [[w, k, *v] for (w, k), v in ST[m].items()] for m in "ng"}}, "E15_chains")
    print(f"鎖 {time.time() - t:.0f}秒")
same = sum(1 for w in L for x in L[w] if x.get("ggenus") and x.get("ggenus") == x.get("genus"))
both = sum(1 for w in L for x in L[w] if x.get("ggenus") and x.get("genus"))
print("親の語が NLI と Gemma で同じ:", same, "/", both)

## 3. 最後の段を聞く（NLI と Gemma、GPU、20〜40分）

In [ ]:
saved = load_gz("E15_arrive")
if saved is not None:
    AR = {(w, k, C): tuple(v) for w, k, C, *v in saved["AR"]}; GA = saved["GA"]
    print("ドライブから読み込み")
else:
    t = time.time()
    REACH = {}
    for sn, its in SETS.items():
        for it in its:
            for mode in "ng":
                REACH[(sn, it["id"], mode)] = reach(L, CH, ST, mode, it["ent"])
    # NLI の着き（今までのやり方）：まだ無い (w,k,C) だけ
    trip = sorted({(w, k, it["catw"]) for sn, its in SETS.items() for it in its for mode in "ng"
                   for (w, k) in REACH[(sn, it["id"], mode)] if w != it["catw"] and (w, k, it["catw"]) not in AR})
    if trip: AR.update(zip(trip, nli(trip)))
    print("NLI の着き 新しく", len(trip))
    # Gemma の着き：path（道を全部）・last（道の最後の説明文だけ）・ctrl（関係のない別の問題の、同じ長さの道）・none（説明文なし、1問1回）
    rng = random.Random(15)
    Q, KEYS = [], []
    for sn, its in SETS.items():
        pool = collections.defaultdict(list)
        for it in its:
            for mode in "ng":
                for n, (p, path) in REACH[(sn, it["id"], mode)].items():
                    pool[len(path)].append((it["ent"], path))
        for it in its:
            x, C = it["ent"], it["catw"]
            Q.append(arrive_prompt(x, C, [])); KEYS.append([sn, it["id"], "-", "none", "", -1])
            for mode in "ng":
                for n, (p, path) in REACH[(sn, it["id"], mode)].items():
                    Q.append(arrive_prompt(x, C, path_lines(L, path))); KEYS.append([sn, it["id"], mode, "path", n[0], n[1]])
                    Q.append(arrive_prompt(x, C, path_lines(L, path[-1:]))); KEYS.append([sn, it["id"], mode, "last", n[0], n[1]])
                    other = [q for e, q in pool[len(path)] if e != x] or [q for ln in pool for e, q in pool[ln] if e != x]
                    Q.append(arrive_prompt(x, C, path_lines(L, rng.choice(other)))); KEYS.append([sn, it["id"], mode, "ctrl", n[0], n[1]])
    # 同じ聞き方は1回だけ聞く
    uq = sorted(set(Q)); print("Gemma に聞く", len(Q), "（重なりを除いて", len(uq), "）")
    V = dict(zip(uq, yes_no(uq)))
    GA = [k + [V[q]] for k, q in zip(KEYS, Q)]
    save_gz({"AR": [[w, k, C, *v] for (w, k, C), v in AR.items()], "GA": GA}, "E15_arrive")
    print(f"着き {time.time() - t:.0f}秒")
save_gz({"items": SETS, "L": L, "CH": {m: [[w, k, v] for (w, k), v in CH[m].items()] for m in "ng"},
         "ST": {m: [[w, k, *v] for (w, k), v in ST[m].items()] for m in "ng"},
         "AR": [[w, k, C, *v] for (w, k, C), v in AR.items()], "GA": GA}, "E15_results")
ex = [g for g in GA if g[0] == "e8" and g[3] == "path"][:5]
for g in ex: print(g)